# 13 — V2 inference / research serving contract: trusted registry, request feature builder, robustness and latency evidence

**สถานะ: งานวิจัยเท่านั้น | production_ready = false | operational = BLOCKED.** ไม่ใช่ระบบเตือนภัยหรือความเสี่ยงรายบุคคล/รายเส้นทาง ไม่มีข้อมูล live

- ไม่ fit, ไม่ tune, ไม่เลือกโมเดลใหม่: ให้บริการเฉพาะ `hgb_poisson__full` (daily) และ `hgb_poisson__calendar_traffic` (6h) ที่ผ่าน gate ใน notebook 11
- เซลล์ `# FUNCTIONS` สองเซลล์ใช้โหลดเป็น service ได้โดยไม่ต้องรันเซลล์รายงาน: `load_registry()` ตรวจ trust anchors ใน `config/inference_v2.json` และ hash ทุกไฟล์ก่อน `joblib.load`;
  `predict_batch(reg, requests)` รับเฉพาะ province_key / target_date / unit / time_block
- feature ของคำขอสร้างจาก FUNCTIONS ของ notebook 10 (ตรวจ hash ก่อน exec) + ตาราง source 09 ที่ตรวจ hash แล้ว ไม่อ่าน panel/label/eligibility
- เอาต์พุต: `reports/v2/serving/*` (aggregate, รายงานไทย, `serving_manifest.json`) และ `data/v2/serving/*` (รายแถว stress, gitignored)

Run All ใน VS Code ด้วย kernel `.venv` หรือ `.venv/bin/python backend/scripts/run_v2_notebooks.py --stage serve-check`

In [1]:
# SETUP: bound numerical thread pools BEFORE numpy / scikit-learn are imported (no fitting happens in 13; prediction is also wrapped in threadpool_limits)
import os
for _k in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS', 'VECLIB_MAXIMUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    os.environ[_k] = '2'
os.environ.setdefault('MPLBACKEND', 'Agg')
print({k: os.environ[k] for k in ('OMP_NUM_THREADS', 'MPLBACKEND')})

{'OMP_NUM_THREADS': '2', 'MPLBACKEND': 'module://matplotlib_inline.backend_inline'}


In [2]:
# FUNCTIONS (notebook 13, part 1: I/O + hashes, pinned execution of the notebook 10/11 FUNCTIONS, trust-anchored verification of every accepted V2 artifact BEFORE joblib.load, version policy, trusted model registry + source cache) - self-contained; a service may exec both FUNCTIONS cells without running any report cell (no network, no fit)
import datetime as dt, gzip, hashlib, json, math, os, sys, time
from pathlib import Path
import numpy as np, pandas as pd

INFERENCE_SCHEMA = 'roadaware-v2-inference-config/1'
SERVING_SCHEMA = 'roadaware-v2-serving-manifest/1'
SERVING_PRODUCER = '13_inference_robustness_contract'
RESPONSE_SCHEMA = 'roadaware-v2-research-prediction/1'
MODEL_MANIFEST_SCHEMA = 'roadaware-v2-model-manifest/1'
MODEL_PRODUCER = '11_v2_model_experiments'
PREP_PRODUCER = '10_v2_preparation_features_leakage'
SELECTION_LOCK_SCHEMA = 'roadaware-v2-selection-lock/1'
ANALYSIS_PRODUCER = '12_v2_evaluation_error_analysis'
RUNNER_LOG_MODEL = 'run_manifest_model.json'
CONFIG_REL = Path('config') / 'inference_v2.json'   # the ONLY configuration a service reads; never a client-supplied path
# every pinned helper 13 calls; exec_pinned_functions returns a plain dict, so ALWAYS use mapping access (fn10[<name>], never attribute access)
FN10_USED = ('read_str_csv', 'weather_features', 'load_calendar_records', 'core_date_features', 'festival_features', 'bank_calendar_features',
             'season_features', 'traffic_vintage_by_date', 'traffic_features', 'validate_feature_columns')
FN11_USED = ('predict_baseline', 'validate_predictions', 'metric_summary')


class DependencyError(RuntimeError):
    """An input is missing, malformed, from another run or differs from the hash its producer recorded: stop, never guess or fall back."""


class RequestValidationError(ValueError):
    """A prediction request was rejected as a whole; .errors = [{'index', 'field', 'code', 'message'}] (index None = batch-level)."""

    def __init__(self, errors):
        self.errors = list(errors)
        super().__init__('; '.join(f"[{e['index']}] {e['field']}: {e['code']} - {e['message']}" for e in self.errors[:10]))


# ---------- generic I/O ----------
def utc_now():
    return dt.datetime.now(dt.timezone.utc).replace(microsecond=0).isoformat()


def sha256_bytes(b):
    return hashlib.sha256(b).hexdigest()


def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''):
            h.update(b)
    return h.hexdigest()


def json_safe(o):
    """Strict-JSON friendly copy: NaN/inf/NaT/NA -> None, numpy scalars -> Python, dates -> ISO strings."""
    if o is None or o is pd.NaT or o is pd.NA:
        return None
    if isinstance(o, dict):
        return {str(k): json_safe(v) for k, v in o.items()}
    if isinstance(o, (list, tuple, set)):
        return [json_safe(v) for v in o]
    if isinstance(o, (bool, np.bool_)):
        return bool(o)
    if isinstance(o, np.integer):
        return int(o)
    if isinstance(o, (float, np.floating)):
        return float(o) if math.isfinite(float(o)) else None
    if isinstance(o, (pd.Timestamp, dt.datetime, dt.date)):
        return o.isoformat()
    if isinstance(o, Path):
        return str(o)
    return o


def write_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + '.tmp')
    tmp.write_text(json.dumps(json_safe(obj), indent=2, ensure_ascii=False, allow_nan=False) + '\n', 'utf-8')
    os.replace(tmp, path)
    return path


def read_json_strict(path):
    return json.loads(Path(path).read_text('utf-8'), parse_constant=lambda c: (_ for _ in ()).throw(ValueError(f'{path}: non-strict JSON {c}')))


def write_csv(path, df):
    """Plain CSV, LF line endings; infinite numeric values are refused."""
    path = Path(path)
    num = df.select_dtypes(include=[np.number])
    if len(num.columns) and np.isinf(num.to_numpy(dtype=float)).any():
        raise ValueError(f'{path.name}: infinite values')
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False, lineterminator='\n')
    return path


def write_csv_gz(path, df):
    """Gzip CSV with a fixed header (mtime 0, no stored name): identical tables give identical bytes."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + '.tmp')
    with open(tmp, 'wb') as fh, gzip.GzipFile(filename='', mode='wb', fileobj=fh, mtime=0) as gz:
        gz.write(df.to_csv(index=False, lineterminator='\n', float_format='%.17g').encode('utf-8'))
    os.replace(tmp, path)
    return path


def verify_file(path, expected_sha256, label):
    """sha256 of an input must equal the recorded hash; raises DependencyError otherwise (missing file included)."""
    p = Path(path)
    if not p.is_file():
        raise DependencyError(f'{label}: missing input {p}')
    got = sha256_file(p)
    if not expected_sha256 or got != expected_sha256:
        raise DependencyError(f'{label}: sha256 {got[:12]} != recorded {str(expected_sha256)[:12]} ({p}); input changed since its producer ran')
    return got


def inside(p, root):
    p, root = Path(p).resolve(), Path(root).resolve()
    return p == root or root in p.parents


def check_write_roots(writes, read_only_dirs, read_only_files):
    """writes: [(label, dir)]; a write root may not equal/contain/sit inside a read-only dir and may not contain any read-only file."""
    for wl, w in writes:
        for rl, r in read_only_dirs:
            if inside(w, r) or inside(r, w):
                raise ValueError(f'{wl} {w} overlaps read-only {rl} {r}')
        hit = [str(f) for f in read_only_files if inside(f, w)]
        if hit:
            raise ValueError(f'{wl} {w} contains read-only inputs {hit[:3]}')
    return True


def find_backend():
    """ROADAWARE_BACKEND_ROOT, else the first cwd ancestor that is (or contains) backend/ with config/inference_v2.json + notebooks/."""
    env = os.environ.get('ROADAWARE_BACKEND_ROOT')
    if env:
        return Path(env).expanduser().resolve()
    here = Path.cwd().resolve()
    for d in (here, *here.parents):
        for cand in (d, d / 'backend'):
            if (cand / CONFIG_REL).is_file() and (cand / 'notebooks').is_dir():
                return cand
    raise FileNotFoundError('backend root not found from the working directory; set ROADAWARE_BACKEND_ROOT')


# ---------- notebook code identity + pinned FUNCTIONS execution ----------
def _code_cells(nb_path):
    nb = json.loads(Path(nb_path).read_text('utf-8'))
    return [(i, c) for i, c in enumerate(nb['cells']) if c['cell_type'] == 'code']


def _src(c):
    return ''.join(c['source']) if isinstance(c['source'], list) else c['source']


def functions_cells_source(nb_path):
    """Concatenated source of every '# FUNCTIONS' code cell in notebook order (same definition notebooks 10/11/12 hash)."""
    hits = [_src(c) for _, c in _code_cells(nb_path) if _src(c).lstrip().startswith('# FUNCTIONS')]
    if not hits:
        raise DependencyError(f'{nb_path}: no FUNCTIONS cell')
    return '\n\n'.join(hits)


def code_cells_sha256(nb_path):
    return {f"{i}:{c.get('id', '')}": sha256_bytes(_src(c).encode('utf-8')) for i, c in _code_cells(nb_path)}


def exec_pinned_functions(nb_path, expected_sha, label, required=(), expected_code_cells=None):
    """Exec ONLY the '# FUNCTIONS' cells of an accepted notebook into a fresh namespace, after their source hash equals the hash its accepted
    run recorded (and, when given, every code cell hash equals the record). The rest of that notebook is never executed.
    Returns the namespace as a plain dict: callers use ns['name'], never attribute access."""
    p = Path(nb_path)
    if not p.is_file():
        raise DependencyError(f'{label}: notebook {p} missing')
    cells = [_src(c) for _, c in _code_cells(p) if _src(c).lstrip().startswith('# FUNCTIONS')]
    got = sha256_bytes('\n\n'.join(cells).encode('utf-8')) if cells else None
    if not expected_sha or got != expected_sha:
        raise DependencyError(f'{label}: FUNCTIONS sha256 {str(got)[:12]} != recorded {str(expected_sha)[:12]}; code changed since the accepted run')
    if expected_code_cells is not None and code_cells_sha256(p) != expected_code_cells:
        raise DependencyError(f'{label}: code cells differ from the accepted run')
    ns = {'__name__': f'roadaware_pinned_{label}'}
    for i, s in enumerate(cells):
        exec(compile(s, f'{label}:FUNCTIONS[{i}]', 'exec'), ns)
    miss = [n for n in required if not callable(ns.get(n))]
    if miss:
        raise DependencyError(f'{label}: pinned FUNCTIONS lack {miss}')
    return ns


# ---------- verification of the accepted 09/10/11/12 artifacts ----------
def verify_runner_log(path, model_rep):
    p = Path(path)
    log = read_json_strict(p)
    nbs = [e for e in log.get('notebooks', []) if e.get('notebook') == MODEL_PRODUCER]
    if log.get('status') != 'ok' or log.get('stage') != 'model' or not nbs or nbs[-1].get('status') != 'ok':
        raise DependencyError(f'runner log {p}: notebook 11 did not finish every cell (status {log.get("status")!r})')
    logged = (log.get('roots') or {}).get('model_reports')
    if not logged or Path(logged).resolve() != Path(model_rep).resolve():
        raise DependencyError(f'runner log {p}: model report dir {logged!r} is not {Path(model_rep).resolve()}')
    return {'path': str(p), 'status': 'ok', 'finished_utc': log.get('finished_utc'), 'cells_completed': nbs[-1].get('cells_completed')}


def verify_model_runner_logs(model_rep):
    """At least one 11 runner log (canonical reports/v2/ or legacy report-local) and every present one ok for this model report dir."""
    rep = Path(model_rep)
    found = [p for p in (rep.parent / RUNNER_LOG_MODEL, rep / RUNNER_LOG_MODEL) if p.is_file()]
    if not found:
        raise DependencyError(f'no notebook 11 runner log next to {rep}: completion of every 11 cell is unproven')
    return [verify_runner_log(p, rep) for p in found]


def verify_model_manifest(man, cfg):
    if man.get('schema') != MODEL_MANIFEST_SCHEMA or man.get('producer') != MODEL_PRODUCER or man.get('status') != 'completed':
        raise DependencyError('model manifest is not a completed notebook 11 run')
    if man.get('production_ready') is not False or man.get('operational_status') != 'BLOCKED':
        raise DependencyError('model manifest must stay research-only (production_ready false, BLOCKED)')
    if not (man.get('selection_lock') or {}).get('verified_after_diagnostics'):
        raise DependencyError('notebook 11 did not verify its selection lock after the diagnostics')
    sel, it = man['selection'], man['next_stage_interface']['selected']
    for u, want in cfg['expected_selection'].items():
        for k, v in want.items():
            if sel.get(u, {}).get(k) != v or it.get(u, {}).get(k) != v:
                raise DependencyError(f'{u}.{k}: manifest {sel.get(u, {}).get(k)!r} != expected {v!r} (another selection; 13 never re-selects)')
        if not sel[u]['gate']['passed']:
            raise DependencyError(f'{u}: locked gate did not pass')
    return True


def verify_report_outputs(man, rep_dir):
    excluded = set(man.get('report_outputs_excluded') or {})
    if excluded - {RUNNER_LOG_MODEL}:
        raise DependencyError(f'model manifest excludes non-runner files {sorted(excluded - {RUNNER_LOG_MODEL})}')
    return {o['path']: verify_file(Path(rep_dir) / o['path'], o['sha256'], f"11 report {o['path']}")
            for o in man['report_outputs'] if o['path'] != RUNNER_LOG_MODEL}


def verify_selection_lock(path, expected_sha, actual_hashes, out11, man):
    p = Path(path)
    verify_file(p, expected_sha, 'selection lock')
    lock = read_json_strict(p)
    if lock.get('schema') != SELECTION_LOCK_SCHEMA:
        raise DependencyError('selection lock schema mismatch')
    for e in lock.get('files', []):
        fp = Path(e['path']) if Path(e['path']).is_absolute() else Path(out11) / e['path']
        verify_file(fp, e['sha256'], f"selection-locked file {e['path']}")
    for k, v in actual_hashes.items():
        if lock.get('hashes', {}).get(k) != v:
            raise DependencyError(f'{k} differs from the selection lock')
    for u, s in lock['units'].items():
        for k in ('recommended_method', 'best_candidate', 'strongest_baseline', 'recommendation_kind'):
            if s.get(k) != man['selection'][u].get(k):
                raise DependencyError(f'{u}.{k} differs between selection lock and model manifest')
    return lock


def recorded_input(pm, role, rel):
    """sha256 notebook 10 recorded for an input of `role` whose path ends with `rel` (exactly one entry)."""
    hits = [i for i in pm['inputs'] if i['role'] == role and Path(i['path']).as_posix().endswith('/' + str(rel).lstrip('/'))]
    if len(hits) != 1:
        raise DependencyError(f'preparation manifest records {len(hits)} {role} inputs ending in {rel}')
    return hits[0]['sha256']


def check_versions(recorded, policy):
    """Runtime packages vs the versions recorded when the bundles were written; raises BEFORE any deserialization on a mismatch."""
    import joblib, sklearn
    have = {'python': sys.version.split()[0], 'numpy': np.__version__, 'pandas': pd.__version__, 'scikit-learn': sklearn.__version__,
            'joblib': joblib.__version__}
    bad = []
    for k in policy['exact']:
        if have[k] != recorded.get(k):
            bad.append(f'{k} {have[k]} != recorded {recorded.get(k)}')
    for k in policy['same_major_minor']:
        if have[k].split('.')[:2] != str(recorded.get(k, '')).split('.')[:2]:
            bad.append(f'{k} {have[k]} not {recorded.get(k)} major.minor')
    if bad:
        raise DependencyError('incompatible runtime for the recorded bundles: ' + '; '.join(bad))
    return have


def verified_joblib_load(path, expected_sha, label, root, loader=None):
    """Bundle path must sit inside the trusted model data root and equal the recorded sha256; only then is it deserialized."""
    p = Path(path)
    if not inside(p, root):
        raise DependencyError(f'{label}: {p} is outside the trusted model data root {root}')
    verify_file(p, expected_sha, label)
    if loader is None:
        import joblib
        loader = joblib.load
    return loader(p)


def resolve_roots(backend, cfg, overrides=None):
    """Trusted input locations from the local inference config (relative to the backend root); explicit overrides come only from the
    notebook/runner (fresh replay or tests), never from a request. Every artifact is still checked against the trust anchors."""
    P = cfg['paths']
    r = {k: (Path(backend) / v).resolve() for k, v in P.items()}
    for k, v in (overrides or {}).items():
        if k not in r:
            raise ValueError(f'unknown root override {k!r}')
        r[k] = Path(v).resolve()
    return r


# ---------- trusted registry ----------
def load_registry(backend=None, cfg=None, roots=None, load_models=True, loader=None):
    """Verify every accepted V2 artifact against the trust anchors in config/inference_v2.json and the hashes each producer recorded, then
    (only then) deserialize the two selected HGB bundles + their strongest baselines and cache the small hash-verified source tables.
    Returns the registry dict used by predict_batch(). Raises DependencyError on any missing/tampered/incompatible input."""
    t0 = time.perf_counter()
    backend = Path(backend).resolve() if backend else find_backend()
    cfg = read_json_strict(backend / CONFIG_REL) if cfg is None else cfg
    if cfg.get('schema') != INFERENCE_SCHEMA or cfg.get('production_ready') is not False or cfg.get('operational_status') != 'BLOCKED':
        raise DependencyError('inference config schema / research status unexpected')
    R = resolve_roots(backend, cfg, roots)
    A = cfg['trust_anchors']
    hashes = {}
    # 11 manifest (anchor), completion, report outputs, inputs, code
    man_path = R['model_report_dir'] / 'model_manifest.json'
    hashes['model_manifest'] = verify_file(man_path, A['model_manifest_sha256'], 'model manifest (trust anchor)')
    man = read_json_strict(man_path)
    verify_model_manifest(man, cfg)
    runner = verify_model_runner_logs(R['model_report_dir'])
    hashes.update({f'11_report:{k}': v for k, v in verify_report_outputs(man, R['model_report_dir']).items()})
    role_paths = {'experiment_lock': R['experiment_lock'], 'protocol_draft': R['protocol_draft'], 'model_config': R['model_config'],
                  'preparation_manifest': R['preparation_report_dir'] / 'preparation_manifest.json', 'feature_config': R['feature_config']}
    rec_inputs = {i['role']: i['sha256'] for i in man['inputs']}
    for role, p in role_paths.items():
        hashes[f'11_input:{role}'] = verify_file(p, rec_inputs.get(role), f'11 input {role}')
    if hashes['11_input:preparation_manifest'] != A['preparation_manifest_sha256'] or hashes['11_input:experiment_lock'] != A['experiment_lock_sha256']:
        raise DependencyError('preparation manifest / experiment lock differ from the trust anchors')
    fn11_sha = man['functions_cells_sha256']
    fn11 = exec_pinned_functions(R['notebook11'], fn11_sha, 'notebook11', sorted(set(cfg['pinned_functions']['notebook11']['required']) | set(FN11_USED)),
                                 man['code_cells_sha256'])
    sel_path = R['model_report_dir'] / man['selection_lock']['path']
    if man['selection_lock']['sha256'] != A['selection_lock_sha256']:
        raise DependencyError('selection lock hash in the model manifest differs from the trust anchor')
    sel = verify_selection_lock(sel_path, A['selection_lock_sha256'],
                                {'experiment_lock_sha256': hashes['11_input:experiment_lock'], 'protocol_sha256': hashes['11_input:protocol_draft'],
                                 'model_config_sha256': hashes['11_input:model_config'],
                                 'preparation_manifest_sha256': hashes['11_input:preparation_manifest'], 'notebook11_functions_sha256': fn11_sha},
                                R['model_report_dir'].parents[2], man)
    hashes['selection_lock'] = A['selection_lock_sha256']
    # 10 manifest, FUNCTIONS, configs, source tables (exactly the bytes 10 used)
    pm = read_json_strict(role_paths['preparation_manifest'])
    if pm.get('producer') != PREP_PRODUCER or pm.get('status') not in ('completed', 'completed_with_gaps') or pm.get('production_ready') is not False:
        raise DependencyError('preparation manifest is not an accepted research-only notebook 10 run')
    if pm['config_sha256'] != hashes['11_input:feature_config']:
        raise DependencyError('feature config differs from the bytes notebook 10 used')
    fn10 = exec_pinned_functions(R['notebook10'], pm['functions_cells_sha256'], 'notebook10',
                                 sorted(set(cfg['pinned_functions']['notebook10']['required']) | set(FN10_USED)))
    hashes['10_functions'] = pm['functions_cells_sha256']
    hashes['11_functions'] = fn11_sha
    hashes['09_config'] = verify_file(R['sources_config'], recorded_input(pm, '09_config', R['sources_config'].name), '09 config')
    src_paths = {}
    for name in cfg['source_tables']['sources_report']:
        src_paths[name] = R['sources_report_dir'] / name
        hashes[f'source:{name}'] = verify_file(src_paths[name], recorded_input(pm, '09_output', name), f'09 output {name}')
    for rel in cfg['source_tables']['external_derived']:
        src_paths[rel] = R['external_root'] / rel
        hashes[f'source:{rel}'] = verify_file(src_paths[rel], recorded_input(pm, '09_derived', rel), f'09 derived {rel}')
    mcfg, fcfg, scfg = (read_json_strict(R[k]) for k in ('model_config', 'feature_config', 'sources_config'))
    # selected bundles + strongest baselines: every hash BEFORE any deserialization
    bundles = {(b['unit'], b['method_id']): b for b in man['bundles']}
    units = {}
    for u, want in cfg['expected_selection'].items():
        mid, bid = want['recommended_method'], want['strongest_baseline']
        if (u, mid) not in bundles or (u, bid) not in bundles:
            raise DependencyError(f'{u}: bundle entry for {mid} / {bid} missing from the model manifest')
        bm, bb = bundles[(u, mid)], bundles[(u, bid)]
        if bm['kind'] != 'sklearn_pipeline_joblib' or bb['kind'] != 'baseline_state_json':
            raise DependencyError(f'{u}: unexpected bundle kinds {bm["kind"]}/{bb["kind"]}')
        mp, bp = R['model_data_dir'] / bm['path'], R['model_data_dir'] / bb['path']
        for p, b, lab in ((mp, bm, f'{u} model bundle {mid}'), (bp, bb, f'{u} baseline state {bid}')):
            if not inside(p, R['model_data_dir']):
                raise DependencyError(f'{lab}: path escapes the model data root')
            hashes[f'bundle:{u}:{b["method_id"]}'] = verify_file(p, b['sha256'], lab)
        meth = man['next_stage_interface']['methods'][u][mid]
        if sorted(meth['feature_columns']) != sorted(bm['feature_columns']):
            raise DependencyError(f'{u}: bundle feature set differs from the method definition')
        units[u] = {'method_id': mid, 'family': meth['family'], 'variant': meth['variant'], 'bundle_path': mp, 'bundle_sha256': bm['sha256'],
                    'feature_columns': list(bm['feature_columns']), 'baseline_method_id': bid, 'baseline_path': bp, 'baseline_sha256': bb['sha256'],
                    'gate': man['selection'][u]['gate'], 'fit_cutoff_local': man['final_fit'][u]['cutoff'], 'n_train': man['final_fit'][u]['n_train'],
                    'max_train_target_end': man['final_fit'][u]['max_train_target_end']}
    runtime = check_versions(man['versions'], cfg['version_policy'])
    threads = int(cfg['compute']['threads'])
    if int(man.get('threads', threads)) != threads:
        raise DependencyError('thread bound differs from the notebook 11 record')
    if load_models:
        cats = list(pm['next_stage_interface']['categorical'])
        for u, e in units.items():
            obj = verified_joblib_load(e['bundle_path'], e['bundle_sha256'], f'{u} model bundle', R['model_data_dir'], loader)
            meta = obj.get('meta', {}) if isinstance(obj, dict) else {}
            if meta.get('method_id') != e['method_id'] or meta.get('unit') != u or list(meta.get('feature_columns_in_order', [])) != e['feature_columns']:
                raise DependencyError(f'{u}: bundle metadata disagrees with the manifest')
            if meta.get('fit_cutoff_local') != e['fit_cutoff_local'] or meta.get('categorical') != [c for c in e['feature_columns'] if c in cats]:
                raise DependencyError(f'{u}: bundle cutoff / categorical columns disagree with the manifest')
            e['pipeline'] = obj['pipeline']
            e['categorical'] = list(meta['categorical'])
            e['numeric'] = list(meta['numeric'])
            e['baseline_state'] = read_json_strict(e['baseline_path'])
            if e['baseline_state'].get('id') != e['baseline_method_id'][len('baseline_'):]:
                raise DependencyError(f'{u}: baseline state id mismatch')
    # small source tables cached once per trusted load (a batch never rereads them)
    P = fcfg['panel']
    points = fn10['read_str_csv'](src_paths['province_points.csv'])
    pts = points[points['province_key'].ne('')]
    provinces = sorted(pts['province_key'].unique())
    if len(provinces) != P['expected_provinces'] or pts['province_key'].duplicated().any():
        raise DependencyError(f'{len(provinces)} province keys in the 09 points; expected {P["expected_provinces"]} unique')
    code_to_key = {int(c): k for c, k in zip(pts['province_code'].astype(int), pts['province_key'])}
    I = dict(issue_hour=P['issue_hour_local'], days_before=P['issue_days_before_target'])
    wb = fcfg['weather']['buffer_hours']
    wx = {}
    for prefix, rel in (('wx_', fcfg['weather']['daily_file']), ('wx_block_', fcfg['weather']['block_file'])):
        raw = pd.read_csv(src_paths[rel], dtype=str, keep_default_na=False)
        feats, audit = fn10['weather_features'](raw, code_to_key, wb, utc_offset=P['utc_offset_seconds'], prefix=prefix, **I)
        wx[prefix] = {'frame': feats, 'audit': audit}
    wxd = wx['wx_']['frame'].drop(columns=['time_block'])
    if wxd.duplicated(['province_key', 'target_date']).any():
        raise DependencyError('daily weather: duplicate province/date rows')
    tsum = fn10['read_str_csv'](src_paths[fcfg['traffic']['summary_file']])
    tmeta = fn10['read_str_csv'](src_paths[fcfg['traffic']['metadata_file']])
    calrec = fn10['load_calendar_records'](src_paths['calendar_records.csv'])
    aliases = dict(scfg['province_aliases']['map'])
    if any(v not in provinces for v in aliases.values()) or any(k in provinces for k in aliases):
        raise DependencyError('province alias map must map non-panel spellings onto panel keys only')
    wvals = {p: f['frame'].dropna(subset=[c for c in f['frame'].columns if c.startswith(p)], how='all') for p, f in wx.items()}
    coverage = {'weather_last_feature_date': {p: (None if not len(v) else str(v['target_date'].max())) for p, v in wvals.items()},
                'weather_feature_years': {p: sorted({int(d[:4]) for d in v['target_date']}) for p, v in wvals.items()},
                'traffic_survey_years_ce': sorted({int(y) for y in tmeta.loc[tmeta['normalize_status'].eq('normalized'), 'survey_year_ce']}),
                'bank_calendar_reviewed_years': sorted({y for s in scfg['calendar']['sources'] if s.get('role') == 'annual_base' for y in s.get('years', [])}),
                'weather_audit': {p: f['audit'] for p, f in wx.items()}}
    reg = {'schema': INFERENCE_SCHEMA, 'backend': backend, 'cfg': cfg, 'roots': R, 'hashes': hashes, 'runner_logs': runner, 'fn10': fn10, 'fn11': fn11,
           'model_manifest': man, 'preparation_manifest': pm, 'selection_lock': sel, 'model_config': mcfg, 'feature_config': fcfg, 'sources_config': scfg,
           'units': units, 'provinces': provinces, 'province_set': set(provinces), 'aliases': aliases, 'panel': P, 'issue': I, 'threads': threads,
           'runtime_versions': runtime, 'recorded_versions': man['versions'], 'coverage': coverage, 'models_loaded': bool(load_models),
           'cache': {'wx_daily': wxd.set_index(['province_key', 'target_date']), 'wx_block': wx['wx_block_']['frame'].set_index(['province_key', 'target_date', 'time_block']),
                     'traffic_summary': tsum, 'traffic_meta': tmeta, 'calendar_records': calrec},
           'loaded_utc': utc_now()}
    reg['load_seconds'] = round(time.perf_counter() - t0, 3)
    reg['registry_id'] = sha256_bytes(json.dumps({k: hashes[k] for k in sorted(hashes)}, sort_keys=True).encode('utf-8'))[:16]
    return reg


def registry_summary(reg):
    """JSON-safe description of the trusted registry (for a metadata endpoint / the serving manifest): exact methods, features, units,
    cutoff, model age reference, data coverage, caveats. No paths to pickles are exposed to clients."""
    man, cfg = reg['model_manifest'], reg['cfg']
    units = {}
    for u, e in reg['units'].items():
        units[u] = {'method_id': e['method_id'], 'family': e['family'], 'variant': e['variant'], 'bundle_sha256': e['bundle_sha256'],
                    'feature_columns_in_order': e['feature_columns'], 'n_features': len(e['feature_columns']),
                    'uses_weather': any(c.startswith('wx_') for c in e['feature_columns']),
                    'uses_traffic': any(c.startswith('trf_') for c in e['feature_columns']),
                    'baseline_comparator': {'method_id': e['baseline_method_id'], 'sha256': e['baseline_sha256']},
                    'training_cutoff_local': e['fit_cutoff_local'], 'max_train_target_end_local': e['max_train_target_end'], 'n_train_cells': e['n_train'],
                    'gate_passed': bool(e['gate']['passed']), 'quantity': 'expected selected recorded MOT-source accident records per cell (Poisson mean)',
                    'cell': 'province x local day' if u == 'daily' else 'province x local 6 h block (00-06, 06-12, 12-18, 18-24)'}
    return json_safe({'schema': INFERENCE_SCHEMA, 'registry_id': reg['registry_id'], 'production_ready': False, 'operational_status': 'BLOCKED',
                      'research_only': True, 'units': units, 'provinces': reg['provinces'], 'province_aliases': reg['aliases'],
                      'request_contract': {k: cfg['request'][k] for k in ('allowed_fields', 'units', 'time_blocks', 'max_batch', 'service_date_range',
                                                                          'service_date_range_note', 'evaluated_period', 'evaluated_period_note', 'timezone',
                                                                          'utc_offset', 'issue_rule')},
                      'data_coverage': {k: v for k, v in reg['coverage'].items() if k != 'weather_audit'},
                      'versions': {'recorded': reg['recorded_versions'], 'runtime': reg['runtime_versions']}, 'threads': reg['threads'],
                      'trust_anchors': {k: v for k, v in cfg['trust_anchors'].items() if k.endswith('sha256')},
                      'caveats': list(reg['feature_config']['readiness']['caveats'][:-1]) + [
                          'no untouched test: 2024 development and 2025 reused diagnostics were both seen during development',
                          'training_eligible is an outcome-derived offline filter; served predictions are returned for every cell regardless of label status',
                          'expected counts are model means of SELECTED RECORDED records, not collision probabilities and not personal or route risk',
                          'daily and 6 h models are independent; the sum of four block predictions need not equal the daily prediction']})

In [3]:
# FUNCTIONS (notebook 13, part 2: strict request validation, leakage-safe request feature builder from the pinned notebook 10 helpers + cached as-of sources, prediction service with labelled baseline comparator, daily/6 h reconciliation diagnostic, stress/benchmark helpers) - self-contained after part 1
import datetime as dt, io, re, time
import numpy as np, pandas as pd

ISO_DATE = re.compile(r'[0-9]{4}-[0-9]{2}-[0-9]{2}')   # ASCII digits only, exactly YYYY-MM-DD (CE)
UNIT_KEYS = {'daily': ['province_key', 'target_date'], '6h': ['province_key', 'target_date', 'time_block']}
QUANTITY = 'expected_recorded_count'
QUANTITY_NOTE = ('model mean of the number of SELECTED RECORDED MOT-source accident records in this province cell; fractional by design; '
                 'not a collision probability, not personal or route risk')


def _err(i, field, code, msg):
    return {'index': i, 'field': field, 'code': code, 'message': msg}


def parse_target_date(v, lo, hi):
    """Strict CE ISO date string inside [lo, hi] -> (date, None) or (None, (code, message)). bool / numbers / NaN / date objects are rejected."""
    if isinstance(v, bool) or not isinstance(v, str):
        return None, ('invalid_type', f'target_date must be a "YYYY-MM-DD" string, got {type(v).__name__}')
    if not ISO_DATE.fullmatch(v):
        return None, ('invalid_format', f'target_date {v!r} is not exactly YYYY-MM-DD (no time, zone, spaces or non-ASCII digits)')
    try:
        d = dt.date.fromisoformat(v)
    except ValueError:
        return None, ('invalid_date', f'target_date {v!r} is not a calendar date')
    if d.year >= 2400:
        return None, ('buddhist_era_year', f'target_date {v!r} looks like a Buddhist Era year; use the CE (Gregorian) year')
    if not (lo <= d <= hi):
        return None, ('out_of_service_range', f'target_date {v} outside the research service range {lo}..{hi} (resource control, not a validated period)')
    return d, None


def validate_requests(reg, requests):
    """Public request contract. requests: list of 1..max_batch dicts holding ONLY province_key, target_date, unit ('daily'|'6h') and, for 6h only,
    time_block. Province keys must be one of the 77 canonical panel keys or an exact audited alias (external_sources_v2.json). Any problem
    rejects the whole batch with RequestValidationError(errors). Returns normalized cells in input order."""
    rc = reg['cfg']['request']
    allowed, units, blocks, mx = set(rc['allowed_fields']), rc['units'], rc['time_blocks'], int(rc['max_batch'])
    lo, hi = (dt.date.fromisoformat(x) for x in rc['service_date_range'])
    if not isinstance(requests, (list, tuple)):
        raise RequestValidationError([_err(None, 'requests', 'invalid_type', 'requests must be a JSON array of objects')])
    if not 1 <= len(requests) <= mx:
        raise RequestValidationError([_err(None, 'requests', 'batch_size', f'batch must hold 1..{mx} requests, got {len(requests)}')])
    errs, cells, seen = [], [], {}
    for i, r in enumerate(requests):
        if not isinstance(r, dict):
            errs.append(_err(i, '(item)', 'invalid_type', 'each request must be an object'))
            continue
        extra = sorted(str(k) for k in r if not isinstance(k, str) or k not in allowed)
        if extra:
            errs.append(_err(i, ','.join(extra), 'unknown_field', f'fields {extra} are not accepted (no outcomes, masks, features, paths or issue times)'))
        n0 = len(errs)
        for f in ('province_key', 'target_date', 'unit'):
            if f not in r:
                errs.append(_err(i, f, 'missing_field', f'{f} is required'))
        unit = r.get('unit')
        if 'unit' in r and (not isinstance(unit, str) or unit not in units):
            errs.append(_err(i, 'unit', 'invalid_unit', f'unit must be one of {units}'))
        pk, canon = r.get('province_key'), None
        if 'province_key' in r:
            if not isinstance(pk, str):
                errs.append(_err(i, 'province_key', 'invalid_type', 'province_key must be a string'))
            elif pk in reg['province_set']:
                canon = pk
            elif pk in reg['aliases']:
                canon = reg['aliases'][pk]
            else:
                errs.append(_err(i, 'province_key', 'unknown_province', f'{pk!r} is not one of the 77 canonical keys or an audited exact alias'))
        d = None
        if 'target_date' in r:
            d, bad = parse_target_date(r['target_date'], lo, hi)
            if bad:
                errs.append(_err(i, 'target_date', *bad))
        tb = r.get('time_block')
        if unit == '6h':
            if 'time_block' not in r:
                errs.append(_err(i, 'time_block', 'missing_field', 'time_block is required for unit 6h'))
            elif not isinstance(tb, str) or tb not in blocks:
                errs.append(_err(i, 'time_block', 'invalid_time_block', f'time_block must be one of {blocks}'))
        elif unit == 'daily' and r.get('time_block') is not None:
            errs.append(_err(i, 'time_block', 'incompatible_unit', 'time_block is only valid with unit 6h'))
        if len(errs) > n0 or extra:
            continue
        key = (canon, d.isoformat(), unit, tb if unit == '6h' else None)
        if key in seen:
            errs.append(_err(i, '(item)', 'duplicate_request', f'same cell as request {seen[key]}'))
            continue
        seen[key] = i
        cells.append({'index': i, 'province_key': canon, 'province_key_input': pk, 'alias_applied': canon != pk, 'target_date': d.isoformat(),
                      'unit': unit, 'time_block': tb if unit == '6h' else None})
    if errs:
        raise RequestValidationError(errs)
    return cells


# ---------- training numeric representation contract ----------
TRAINING_NUMERIC_CONTRACT = (
    'the bundles were fitted on numeric features that crossed a CSV text boundary: accepted notebook 10 write_csv_gz wrote them with '
    "DataFrame.to_csv(index=False, lineterminator='\\n') and pandas DEFAULT float text (no float_format), and accepted notebook 11 "
    "load_unit_frame parsed them with pd.read_csv(keep_default_na=False, na_values=['']) and the DEFAULT C float converter (not round_trip), "
    'then pd.to_numeric(errors=raise).astype(float). That converter can land 1 ulp away from the in-memory value (seen on doy_sin / doy_cos, '
    '|diff| 1.1e-16), and a tree threshold sitting between the two values changes the prediction. Request features therefore pass the '
    'same text + parser round trip before prediction: no rounding to a chosen number of decimals, no lookup of stored features, no outcome.')


def training_numeric_representation(frame, numeric_cols):
    """Return a copy of frame whose numeric_cols hold exactly the float64 values notebook 11 trained on for the same in-memory features
    (TRAINING_NUMERIC_CONTRACT): float64 -> pandas default to_csv text -> pandas default read_csv parse -> float64. NaN stays NaN ('' on
    both sides); every other column, the row order and the column order are untouched. Raises on non-numeric or infinite values. Idempotent,
    deterministic and independent of the batch: each value is written and parsed on its own, so the result is the same for future dates."""
    cols = list(numeric_cols)
    if len(set(cols)) != len(cols) or any(c not in frame.columns for c in cols):
        raise ValueError('training_numeric_representation: numeric columns must be unique and present')
    out = frame.copy()
    if not cols:
        return out
    num = pd.DataFrame({c: pd.to_numeric(frame[c], errors='raise').astype(float).to_numpy() for c in cols}, columns=cols)
    if np.isinf(num.to_numpy(dtype=float)).any():
        raise ValueError('training_numeric_representation: infinite feature value')
    back = pd.read_csv(io.StringIO(num.to_csv(index=False, lineterminator='\n')), keep_default_na=False, na_values=[''])
    if list(back.columns) != cols or len(back) != len(frame):
        raise AssertionError('training_numeric_representation: CSV round trip changed the columns or the number of rows')
    for c in cols:
        out[c] = pd.to_numeric(back[c], errors='raise').astype(float).to_numpy()
    return out


# ---------- leakage-safe request feature builder ----------
def build_feature_frame(reg, cells, unit):
    """Features for arbitrary (province_key, target_date[, time_block]) cells in the trained numeric representation (see
    TRAINING_NUMERIC_CONTRACT): build_raw_feature_frame, then training_numeric_representation over every approved non-categorical feature of
    the unit. Categoricals, keys and audit columns are left as built. This is the only feature path used for prediction."""
    raw = build_raw_feature_frame(reg, cells, unit)
    cats = set(reg['preparation_manifest']['next_stage_interface']['categorical'])
    num = [f['name'] for f in reg['feature_config']['features'] if unit in f['panels'] and f['name'] not in cats and f['name'] not in UNIT_KEYS[unit]]
    return training_numeric_representation(raw, num)


def build_raw_feature_frame(reg, cells, unit):
    """In-memory features (BEFORE the training numeric representation; diagnostics only, never fed to a bundle directly) for arbitrary
    (province_key, target_date[, time_block]) cells WITHOUT panels, labels or eligibility: pinned notebook 10 date /
    festival / season helpers, as-of bank calendar replay, newest PRIOR traffic vintage available at issue, archived weather only where the
    notebook 10 guard passed (cached, hash-verified). Missing sources stay NaN (never 0, never a forecast). Returns rows in input order with
    every approved feature of the unit plus audit columns (bank_calendar_coverage, trf_vintage_*); does not check the service date range."""
    fn10, fc, sc, P, I, C = reg['fn10'], reg['feature_config'], reg['sources_config'], reg['panel'], reg['issue'], reg['cache']
    keys = UNIT_KEYS[unit]
    base = pd.DataFrame(cells)[keys].reset_index(drop=True).astype(str)
    if not base['province_key'].isin(reg['province_set']).all():
        raise ValueError('build_feature_frame: non-canonical province key')
    base['_i'] = np.arange(len(base))
    dates = sorted(base['target_date'].unique())
    F = fc['festivals']
    D = fn10['core_date_features'](dates)
    D = D.merge(fn10['festival_features'](dates, sc['calendar']['recurring_fixed_festivals'], F['clip_days'], F['windows']), on='target_date')
    bank = fn10['bank_calendar_features'](dates, C['calendar_records'], sc['calendar']['sources'], **I)
    D = D.merge(bank[['target_date', 'bank_holiday_asof', 'bank_calendar_known_asof', 'bank_calendar_coverage']], on='target_date')
    D = D.merge(fn10['season_features'](dates, fc['season']['months']), on='target_date')
    tv = fn10['traffic_vintage_by_date'](C['traffic_meta'], dates, utc_offset=P['utc_offset_seconds'], **I)
    trf = fn10['traffic_features'](C['traffic_summary'], tv, sorted(base['province_key'].unique()))
    out = base.merge(D, on='target_date', how='left').merge(trf, on=['province_key', 'target_date'], how='left')
    wd = C['wx_daily'].reindex(pd.MultiIndex.from_frame(out[['province_key', 'target_date']]))
    for c in wd.columns:
        out[c] = wd[c].to_numpy(dtype=float)
    if unit == '6h':
        wb = C['wx_block'].reindex(pd.MultiIndex.from_frame(out[['province_key', 'target_date', 'time_block']]))
        for c in wb.columns:
            out[c] = wb[c].to_numpy(dtype=float)
    out['province'] = out['province_key']
    allow = [f['name'] for f in fc['features'] if unit in f['panels']]
    for c in allow:
        if c not in out:
            out[c] = np.nan
    fn10['validate_feature_columns'](allow, allow, fc['forbidden_feature_names'], fc['forbidden_feature_substrings'])
    out = out.sort_values('_i', kind='stable').reset_index(drop=True)
    if len(out) != len(base):
        raise AssertionError('feature builder changed the number of rows')
    return out


def model_matrix(frame, e):
    """Exact trained column order of the bundle; categoricals as strings (as the encoder saw them), numerics float with NaN kept."""
    X = {}
    for c in e['feature_columns']:
        if c in e['categorical']:
            X[c] = frame[c].astype(str).to_numpy(dtype=object)
        else:
            v = pd.to_numeric(frame[c], errors='raise').to_numpy(dtype=float)
            if np.isinf(v).any():
                raise ValueError(f'{c}: infinite feature value')
            X[c] = v
    return pd.DataFrame(X, columns=e['feature_columns'])


def baseline_frame(frame):
    td = pd.to_datetime(frame['target_date'], format='%Y-%m-%d')
    out = pd.DataFrame({'province_key': frame['province_key'].astype(str).to_numpy(), 'bl_weekday': td.dt.dayofweek.astype(str).to_numpy(),
                        'bl_month': td.dt.month.astype(str).to_numpy()})
    if 'time_block' in frame:
        out['time_block'] = frame['time_block'].astype(str).to_numpy()
    return out


def predict_frame(reg, frame, unit, baseline=True):
    """Selected bundle prediction (+ strongest-baseline comparator) for a feature frame; finite >= 0 enforced by the pinned 11 validator; no
    fallback model on error."""
    from threadpoolctl import threadpool_limits
    e = reg['units'][unit]
    X = model_matrix(frame, e)
    with threadpool_limits(limits=reg['threads']):
        mu = e['pipeline'].predict(X)
    mu = reg['fn11']['validate_predictions'](mu, f"{unit} {e['method_id']}")
    b = reg['fn11']['validate_predictions'](reg['fn11']['predict_baseline'](e['baseline_state'], baseline_frame(frame)), e['baseline_method_id']) if baseline else None
    return mu, b


def cell_times(date_str, unit, block, P):
    off = reg_offset(P)
    d = dt.date.fromisoformat(date_str)
    issue = dt.datetime.combine(d - dt.timedelta(days=int(P['issue_days_before_target'])), dt.time(int(P['issue_hour_local'])))
    if unit == 'daily':
        s, e = dt.datetime.combine(d, dt.time(0)), dt.datetime.combine(d + dt.timedelta(days=1), dt.time(0))
    else:
        h = int(block[:2])
        s = dt.datetime.combine(d, dt.time(0)) + dt.timedelta(hours=h)
        e = s + dt.timedelta(hours=int(P['block_hours']))
    f = lambda x: x.strftime('%Y-%m-%dT%H:%M:%S') + off
    return {'issue_time_local': f(issue), 'target_start_local': f(s), 'target_end_local': f(e), 'timezone': P['timezone'], 'utc_offset': off}


def reg_offset(P):
    s = int(P['utc_offset_seconds'])
    return f"{'+' if s >= 0 else '-'}{abs(s) // 3600:02d}:{abs(s) % 3600 // 60:02d}"


def row_limitations(reg, unit, row):
    """Explicit coverage / period caveats of ONE served cell (no risk colour, no percentage)."""
    rc, lim = reg['cfg']['request'], reg['cfg']['limitations']
    e = reg['units'][unit]
    d = dt.date.fromisoformat(row['target_date'])
    ev_lo, ev_hi = (dt.date.fromisoformat(x) for x in rc['evaluated_period'])
    out = ['research_only_not_operational', 'no_untouched_test']
    out.append('within_evaluated_period_previously_seen_diagnostic' if ev_lo <= d <= ev_hi else 'out_of_evaluated_period')
    if d > dt.date.fromisoformat(lim['stale_model_after']):
        out.append('stale_model')
    wx_cols = [c for c in e['feature_columns'] if c.startswith('wx_')]
    if wx_cols and any(pd.isna(row[c]) for c in wx_cols):
        out.append('weather_covariates_missing')
    trf_cols = [c for c in e['feature_columns'] if c.startswith('trf_')]
    if trf_cols and any(pd.isna(row[c]) for c in trf_cols):
        out.append('traffic_covariates_missing')
    elif trf_cols and not pd.isna(row['trf_vintage_survey_year_ce']) and d.year - int(row['trf_vintage_survey_year_ce']) > int(lim['traffic_stale_after_years']):
        out.append('traffic_vintage_stale')
    if int(row['bank_calendar_known_asof']) == 0:
        out.append('bank_calendar_unknown')
    return out


def make_result(reg, cell, row, mu, base):
    e, P = reg['units'][cell['unit']], reg['panel']
    cutoff = dt.date.fromisoformat(e['fit_cutoff_local'][:10])
    vint = None if pd.isna(row['trf_vintage_survey_year_ce']) else int(row['trf_vintage_survey_year_ce'])
    wx_cols = [c for c in reg['preparation_manifest']['next_stage_interface']['feature_groups']['weather'] if c in row.index]
    return json_safe({
        'index': cell['index'],
        'request': {'province_key': cell['province_key_input'], 'target_date': cell['target_date'], 'unit': cell['unit'], 'time_block': cell['time_block']},
        'province_key': cell['province_key'], 'alias_applied': cell['alias_applied'], 'target_date': cell['target_date'], 'unit': cell['unit'],
        'time_block': cell['time_block'], 'cell': cell_times(cell['target_date'], cell['unit'], cell['time_block'], P),
        'prediction': {QUANTITY: float(mu), 'quantity_note': QUANTITY_NOTE, 'method_id': e['method_id'], 'family': e['family'], 'variant': e['variant'],
                       'bundle_sha256': e['bundle_sha256'], 'registry_id': reg['registry_id']},
        'baseline_comparator': None if base is None else {QUANTITY: float(base), 'method_id': e['baseline_method_id'], 'sha256': e['baseline_sha256'],
                                                           'label': 'strongest notebook-11 baseline; reference comparator only, NOT the selected model'},
        'source_coverage': {'weather_used_by_model': any(c.startswith('wx_') for c in e['feature_columns']),
                            'weather_values_available': bool(wx_cols) and not all(pd.isna(row[c]) for c in wx_cols),
                            'weather_last_feature_date': reg['coverage']['weather_last_feature_date']['wx_'],
                            'traffic_used_by_model': any(c.startswith('trf_') for c in e['feature_columns']),
                            'traffic_vintage_survey_year_ce': vint, 'traffic_vintage_lag_years': None if vint is None else int(cell['target_date'][:4]) - vint,
                            'bank_calendar_coverage': row['bank_calendar_coverage'], 'bank_calendar_known_asof': int(row['bank_calendar_known_asof']),
                            'missing_covariates_policy': 'missing external covariates stay NaN and are handled natively by HGB; no forecast or backfill is fabricated'},
        'model_age': {'training_cutoff_local': e['fit_cutoff_local'], 'days_from_training_cutoff_to_target': (dt.date.fromisoformat(cell['target_date']) - cutoff).days},
        'limitations': row_limitations(reg, cell['unit'], row),
        'research_status': {'research_only': True, 'production_ready': False, 'operational_status': 'BLOCKED', 'live_data': False},
    })


def predict_batch(reg, requests, include_baseline=True):
    """PUBLIC SERVICE FUNCTION. predict_batch(reg, requests) -> {'schema', 'registry_id', ..., 'results': [one per request, input order]}.
    reg comes from load_registry(); requests follow validate_requests(). Raises RequestValidationError (client error, whole batch) or
    DependencyError / ValueError (server-side: the registry or a prediction is invalid; never silently switches model)."""
    if not reg.get('models_loaded'):
        raise DependencyError('registry was loaded without models')
    t0 = time.perf_counter()
    cells = validate_requests(reg, requests)
    results = [None] * len(requests)
    for unit in [u for u in reg['cfg']['request']['units'] if any(c['unit'] == u for c in cells)]:
        sub = [c for c in cells if c['unit'] == unit]
        fr = build_feature_frame(reg, sub, unit)
        mu, base = predict_frame(reg, fr, unit, include_baseline)
        for j, c in enumerate(sub):
            results[c['index']] = make_result(reg, c, fr.iloc[j], mu[j], None if base is None else base[j])
    if any(r is None for r in results):
        raise AssertionError('a request has no result')
    return {'schema': RESPONSE_SCHEMA, 'registry_id': reg['registry_id'], 'research_only': True, 'production_ready': False,
            'operational_status': 'BLOCKED', 'n_results': len(results), 'seconds': round(time.perf_counter() - t0, 6), 'results': results}


def daily_block_reconciliation(reg, province_keys, dates):
    """Diagnostic only: independent daily model vs the sum of the four independent 6 h block predictions for the same province-days (not forced equal)."""
    cd = [{'province_key': p, 'target_date': d} for p in province_keys for d in dates]
    cb = [dict(c, time_block=b) for c in cd for b in reg['cfg']['request']['time_blocks']]
    md, _ = predict_frame(reg, build_feature_frame(reg, cd, 'daily'), 'daily', baseline=False)
    mb, _ = predict_frame(reg, build_feature_frame(reg, cb, '6h'), '6h', baseline=False)
    out = pd.DataFrame(cd)
    out['daily_pred'] = md
    out['sum_block_pred'] = mb.reshape(-1, 4).sum(axis=1)
    out['diff_daily_minus_blocks'] = out['daily_pred'] - out['sum_block_pred']
    out['ratio_blocks_to_daily'] = out['sum_block_pred'] / out['daily_pred']
    return out


# ---------- evaluation helpers (notebook only) ----------
def load_feature_rows(path, keys, cols, cats):
    """Accepted notebook 10 feature matrix rows (string keys/categoricals, float numerics, NaN kept)."""
    str_cols = list(dict.fromkeys(list(keys) + [c for c in cats if c in cols]))
    fe = pd.read_csv(path, dtype={c: str for c in str_cols}, keep_default_na=False, na_values=[''])
    miss = [c for c in list(keys) + list(cols) if c not in fe.columns]
    if miss:
        raise DependencyError(f'{Path(path).name}: missing columns {miss}')
    for c in cols:
        if c not in str_cols:
            fe[c] = pd.to_numeric(fe[c], errors='raise').astype(float)
    return fe


def compare_features(built, ref, cols, cats, tol):
    """Per-column parity: categoricals exact, numerics |diff| <= tol with identical NaN pattern. n_not_bitwise_equal (reported, not a
    tolerance) counts numeric rows whose float64 values are not identical (NaN == NaN), the boundary a tree split can amplify."""
    rows = []
    for c in cols:
        if c in cats:
            eq = built[c].astype(str).to_numpy() == ref[c].astype(str).to_numpy()
            rows.append({'feature': c, 'n': len(eq), 'n_mismatch': int((~eq).sum()), 'max_abs_diff': None, 'nan_pattern_equal': True,
                         'n_not_bitwise_equal': int((~eq).sum())})
            continue
        a, b = pd.to_numeric(built[c]).to_numpy(float), pd.to_numeric(ref[c]).to_numpy(float)
        na, nb = np.isnan(a), np.isnan(b)
        both = ~na & ~nb
        d = np.abs(a[both] - b[both])
        rows.append({'feature': c, 'n': len(a), 'n_nan_ref': int(nb.sum()), 'nan_pattern_equal': bool((na == nb).all()),
                     'max_abs_diff': float(d.max()) if d.size else 0.0, 'n_mismatch': int((na != nb).sum() + (d > tol).sum()),
                     'n_not_bitwise_equal': int((na != nb).sum() + (a[both] != b[both]).sum())})
    return pd.DataFrame(rows)


def apply_missing(frame, groups, feature_groups):
    """Stress copy: every column of the named feature groups set to NaN (a missing-input perturbation, not an alternate world)."""
    out = frame.copy()
    for g in groups:
        for c in feature_groups[g]:
            if c in out:
                out[c] = np.nan
    return out


def timing_summary(seconds):
    s = np.asarray(seconds, dtype=float)
    return {'n': int(s.size), 'median_ms': float(np.median(s) * 1e3), 'p95_ms': float(np.percentile(s, 95) * 1e3), 'min_ms': float(s.min() * 1e3),
            'max_ms': float(s.max() * 1e3)}

## A. Trusted registry และ guard

ตรวจ anchors → manifest/runner log/รายงาน/อินพุต/โค้ด 11 → selection lock → FUNCTIONS 10/11 (exec เฉพาะเซลล์ที่ hash ตรง) → ตาราง source 09 → hash ของ bundle → เวอร์ชัน แล้วจึง deserialize

In [4]:
# A. Roots, trusted registry (cold load: every hash verified BEFORE joblib.load), write guards and read-only snapshot
import sklearn, joblib, threadpoolctl, scipy
T_RUN0 = utc_now()
BACKEND = find_backend()
REPO = BACKEND.parent
CFG_PATH = BACKEND / CONFIG_REL
CFG = read_json_strict(CFG_PATH)
NB13 = BACKEND / 'notebooks' / '13_inference_robustness_contract.ipynb'
OUT = Path(os.environ.get('ROADAWARE_V2_OUTPUT_ROOT') or BACKEND).resolve()                       # 13 writes only below OUT
_ENV_ROOTS = {'model_report_dir': 'ROADAWARE_V2_MODEL_REPORT_DIR', 'model_data_dir': 'ROADAWARE_V2_MODEL_DATA_ROOT',
              'preparation_report_dir': 'ROADAWARE_V2_PREPARATION_DIR', 'preparation_data_dir': 'ROADAWARE_V2_DATA_ROOT',
              'sources_report_dir': 'ROADAWARE_V2_SOURCES_DIR', 'external_root': 'ROADAWARE_V2_EXTERNAL_ROOT',
              'analysis_report_dir': 'ROADAWARE_V2_ANALYSIS_REPORT_DIR', 'experiment_lock': 'ROADAWARE_V2_EXPERIMENT_LOCK'}
ROOT_OVERRIDES = {k: os.environ[v] for k, v in _ENV_ROOTS.items() if os.environ.get(v)}   # runner-set read-only roots (fresh replay)
REP = OUT / CFG['outputs']['report_subdir']
SDATA = Path(os.environ.get('ROADAWARE_V2_SERVING_DATA_ROOT') or OUT / CFG['outputs']['data_subdir']).resolve()
V1 = Path(os.environ.get('ROADAWARE_V1_DATA_ROOT') or BACKEND).resolve()
V1_PROTECTED = [V1 / 'data' / r for r in ('raw', 'interim', 'processed', 'model')] + [V1 / 'reports' / r for r in ('data', 'models')]
CHECKS = []  # only checks that actually executed are appended


def passed(name, detail=''):
    CHECKS.append({'check': name, 'status': 'passed', 'detail': detail})
    print('PASS', name, detail)


if (REP / 'serving_manifest.json').exists():
    raise RuntimeError(f'{REP / "serving_manifest.json"} already exists (earlier notebook 13 run, preserved). Use a fresh root, e.g.\n'
                       '  .venv/bin/python backend/scripts/run_v2_notebooks.py --stage serve-check --output-dir backend/rerun-serving-<UTC-stamp>\n'
                       'or in VS Code set ROADAWARE_V2_OUTPUT_ROOT=<fresh dir> before Run All.')
_t = time.perf_counter()
REG = load_registry(BACKEND, CFG, ROOT_OVERRIDES)
COLD_LOAD_S = time.perf_counter() - _t
R = REG['roots']
passed('trusted_registry_loaded', f"registry {REG['registry_id']}: {len(REG['hashes'])} hashes verified (anchors, 11 reports/inputs/code, selection lock + locked files, "
       f"10 FUNCTIONS, 09 sources, bundles) before deserializing {sorted(e['method_id'] for e in REG['units'].values())}; cold load {COLD_LOAD_S:.2f}s")
passed('versions_compatible', f"runtime {REG['runtime_versions']} vs recorded {REG['recorded_versions']} (exact scikit-learn/joblib)")
UNITS = list(CFG['expected_selection'])
MAN, PM = REG['model_manifest'], REG['preparation_manifest']
IT, IF10 = MAN['next_stage_interface'], PM['next_stage_interface']
PM_SHA = {o['path']: o['sha256'] for o in PM['data_outputs']}
FEAT_FILES, DIAG_FILES = {}, {}
for u in UNITS:
    pf = IT['panels_and_features'][u]
    for part in ('panel', 'features'):
        if pf[f'{part}_sha256'] != PM_SHA.get(pf[part]):
            raise DependencyError(f'{u} {part}: 11 interface hash differs from the 10 manifest')
        FEAT_FILES[(u, part)] = R['preparation_data_dir'] / pf[part]
        verify_file(FEAT_FILES[(u, part)], PM_SHA[pf[part]], f'10 {u} {part}')
    d = IT['diagnostic_predictions'][u]
    DIAG_FILES[u] = R['model_data_dir'] / d['path']
    verify_file(DIAG_FILES[u], d['sha256'], f'11 diagnostic predictions {u}')
passed('evaluation_inputs_hashed', '10 panels/features and 11 diagnostic predictions equal their recorded hashes')

READ_ONLY_FILES = [CFG_PATH, R['notebook10'], R['notebook11'], R['model_report_dir'] / 'model_manifest.json',
                   R['analysis_report_dir'] / 'analysis_manifest.json'] + list(FEAT_FILES.values()) + list(DIAG_FILES.values()) + \
                  [e[k] for e in REG['units'].values() for k in ('bundle_path', 'baseline_path')]
check_write_roots([('serving report dir', REP), ('serving data root', SDATA)],
                  [(k, R[k]) for k in ('model_report_dir', 'model_data_dir', 'preparation_report_dir', 'sources_report_dir', 'external_root',
                                       'analysis_report_dir')] + [('protected v1', v) for v in V1_PROTECTED], READ_ONLY_FILES)
if inside(R['preparation_data_dir'], SDATA) or inside(R['preparation_data_dir'], REP):
    raise ValueError('serving output root equals or contains the read-only 10 data root')
SNAP_BEFORE = {str(p): sha256_file(p) for p in READ_ONLY_FILES}
passed('read_only_snapshot_before', f'{len(SNAP_BEFORE)} read-only files hashed; outputs -> {REP} and {SDATA}')
REP.mkdir(parents=True, exist_ok=True)
SDATA.mkdir(parents=True, exist_ok=True)
WRITTEN = []


def out_csv(name, df, gz=False):
    p = write_csv_gz(SDATA / name, df) if gz else write_csv(REP / name, df)
    WRITTEN.append(p)
    return p


EV = CFG['evaluation']
FLOOR = float(IT['deviance_floor'])
FN11 = REG['fn11']
VERSIONS = dict(REG['runtime_versions'], scipy=scipy.__version__, threadpoolctl=threadpoolctl.__version__)
pd.set_option('display.width', 200)
print(json.dumps(registry_summary(REG)['units'], indent=1, ensure_ascii=False)[:3000])

PASS trusted_registry_loaded registry 2f8a1e2e910fcd01: 32 hashes verified (anchors, 11 reports/inputs/code, selection lock + locked files, 10 FUNCTIONS, 09 sources, bundles) before deserializing ['hgb_poisson__calendar_traffic', 'hgb_poisson__full']; cold load 1.60s
PASS versions_compatible runtime {'python': '3.9.6', 'numpy': '2.0.2', 'pandas': '2.3.3', 'scikit-learn': '1.6.1', 'joblib': '1.5.3'} vs recorded {'python': '3.9.6', 'numpy': '2.0.2', 'pandas': '2.3.3', 'scikit-learn': '1.6.1', 'scipy': '1.13.1', 'joblib': '1.5.3', 'threadpoolctl': '3.7.0'} (exact scikit-learn/joblib)
PASS evaluation_inputs_hashed 10 panels/features and 11 diagnostic predictions equal their recorded hashes
PASS read_only_snapshot_before 15 read-only files hashed; outputs -> /Users/earthnaja/Documents/GitHub/Work - Earth/Project Work/Source Code/RoadAware/backend/reports/v2/serving and /Users/earthnaja/Documents/GitHub/Work - Earth/Project Work/Source Code/RoadAware/backend/data/v2/serving
{
 "daily": {
  "

## B. Parity

โหลดซ้ำเทียบ prediction ที่ 11 บันทึกไว้ (ทุกแถว diagnostic) ทั้งจาก feature ของ 10 และจาก request builder; builder เทียบตาราง feature 10 บนตัวอย่างกำหนดแน่นอน (leap day, ข้ามปี, เทศกาล, ครบ 4 ช่วง, source ขาด, eligible และ unknown-label)

In [5]:
# B. Reload parity vs the saved notebook 11 diagnostics (ALL rows), request-builder parity vs the accepted notebook 10 feature tables
CATS = list(IF10['categorical'])
ALLOW = {u: [f['name'] for f in REG['feature_config']['features'] if u in f['panels']] for u in UNITS}
TOL_R, TOL_B = float(EV['reload_parity_max_abs_diff']), float(EV['builder_parity_max_abs_diff'])
DIAG, REF, PARITY, FEAT_PARITY, NUMREP = {}, {}, [], [], []
for u in UNITS:
    keys, e = UNIT_KEYS[u], REG['units'][u]
    dg = pd.read_csv(DIAG_FILES[u], dtype={k: str for k in keys}, keep_default_na=False, na_values=[''])
    for m in (e['method_id'], e['baseline_method_id']):
        dg[m] = pd.to_numeric(dg[m], errors='raise').astype(float)
    REF[u] = load_feature_rows(FEAT_FILES[(u, 'features')], keys, [c for c in ALLOW[u] if c not in keys], CATS)
    j = dg[keys].merge(REF[u], on=keys, how='left', validate='one_to_one', indicator=True)
    if not j['_merge'].eq('both').all():
        raise DependencyError(f'{u}: diagnostic keys absent from the 10 features')
    j = j.drop(columns='_merge')
    DIAG[u] = dg
    # (1) saved 10 features -> reloaded bundle / baseline state
    mu_ref, b_ref = predict_frame(REG, j, u)
    # (2) request builder (no panel / label / eligibility) -> same bundle
    built = build_feature_frame(REG, dg[keys].to_dict('records'), u)
    mu_b, b_b = predict_frame(REG, built, u)
    for path, mu, b in (('saved_10_features', mu_ref, b_ref), ('request_builder', mu_b, b_b)):
        for kind, m, pred in (('selected', e['method_id'], mu), ('baseline_comparator', e['baseline_method_id'], b)):
            for sp, idx in [('all', np.arange(len(dg)))] + [(s, np.flatnonzero(dg['split'].eq(s).to_numpy())) for s in sorted(dg['split'].unique())]:
                d = np.abs(pred[idx] - dg[m].to_numpy()[idx])
                PARITY.append({'unit': u, 'feature_path': path, 'role': kind, 'method_id': m, 'split': sp, 'n': int(len(idx)),
                               'max_abs_diff': float(d.max()) if len(d) else 0.0, 'tolerance': TOL_R, 'pass': bool((d <= TOL_R).all())})
    fp = compare_features(built, j, [c for c in ALLOW[u] if c not in keys], CATS, TOL_B).assign(unit=u, sample='all_diagnostic_rows')
    FEAT_PARITY.append(fp)
    DIAG[u + ':built'] = built
    # (3) evidence for the training numeric representation contract: in-memory builder output BEFORE the 10 -> 11 text/parse round trip
    #     (diagnostic only; prediction never uses it). Reported, not gated: the gate is (2) above.
    raw = build_raw_feature_frame(REG, dg[keys].to_dict('records'), u)
    mu_raw, _ = predict_frame(REG, raw, u, baseline=False)
    num_cols = [c for c in ALLOW[u] if c not in keys and c not in CATS]
    fr_raw = compare_features(raw, j, num_cols, CATS, TOL_B).set_index('feature')
    fr_rep = compare_features(raw, built, num_cols, CATS, TOL_B).set_index('feature')
    fr_new = fp.set_index('feature')
    for c in num_cols:
        NUMREP.append({'unit': u, 'feature': c, 'n': int(len(dg)), 'n_values_changed_by_contract': int(fr_rep.loc[c, 'n_not_bitwise_equal']),
                       'max_abs_change_by_contract': fr_rep.loc[c, 'max_abs_diff'], 'n_not_bitwise_equal_10_raw_builder': int(fr_raw.loc[c, 'n_not_bitwise_equal']),
                       'n_not_bitwise_equal_10_request_builder': int(fr_new.loc[c, 'n_not_bitwise_equal']),
                       'max_abs_diff_10_request_builder': fr_new.loc[c, 'max_abs_diff']})
    NUMREP.append({'unit': u, 'feature': '(selected prediction vs saved 11)', 'n': int(len(dg)), 'n_values_changed_by_contract': int((mu_raw != mu_b).sum()),
                   'max_abs_change_by_contract': float(np.abs(mu_raw - mu_b).max()),
                   'n_not_bitwise_equal_10_raw_builder': int((mu_raw != dg[e['method_id']].to_numpy()).sum()),
                   'n_not_bitwise_equal_10_request_builder': int((mu_b != dg[e['method_id']].to_numpy()).sum()),
                   'max_abs_diff_10_request_builder': float(np.abs(mu_b - dg[e['method_id']].to_numpy()).max()),
                   'max_abs_diff_raw_builder_vs_saved_11': float(np.abs(mu_raw - dg[e['method_id']].to_numpy()).max())})
    del raw, mu_raw
PARITY = pd.DataFrame(PARITY)
out_csv('reload_parity.csv', PARITY)
NUMREP = pd.DataFrame(NUMREP)
out_csv('training_numeric_representation.csv', NUMREP)
assert PARITY['pass'].all(), PARITY[~PARITY['pass']]
passed('reload_parity_all_diagnostic_rows', '; '.join(f"{u}: {len(DIAG[u])} rows, max |diff| selected "
       f"{PARITY[(PARITY.unit == u) & (PARITY.role == 'selected')]['max_abs_diff'].max():.2e}" for u in UNITS) + f' (tol {TOL_R})')
_cyc = NUMREP[NUMREP['feature'].isin(['doy_sin', 'doy_cos'])]
assert len(_cyc) == 2 * len(UNITS) and _cyc['n_not_bitwise_equal_10_request_builder'].eq(0).all(), _cyc
_feat_rows = NUMREP[~NUMREP['feature'].str.startswith('(')]
NUMREP_SUM = {u: {'n_feature_values_changed_by_contract': int(g['n_values_changed_by_contract'].sum()),
                  'n_feature_values_not_bitwise_equal_to_10': int(g['n_not_bitwise_equal_10_request_builder'].sum()),
                  'max_abs_diff_raw_builder_vs_saved_11': float(NUMREP[(NUMREP.unit == u) & NUMREP.feature.str.startswith('(')]['max_abs_diff_raw_builder_vs_saved_11'].iloc[0])}
              for u, g in _feat_rows.groupby('unit', sort=False)}
passed('training_numeric_representation', 'request features use the 10 -> 11 default CSV text/parse representation (TRAINING_NUMERIC_CONTRACT); '
       f'doy_sin/doy_cos bitwise equal to the saved 10 features on every diagnostic row; {NUMREP_SUM}')

# deterministic builder sample over the WHOLE 10 grid (2021-2025, eligible AND unknown-label cells; panel read for annotation only)
SPECIAL_DATES = ['2021-01-01', '2022-12-31', '2023-01-01', '2023-04-14', '2023-12-31', '2024-01-01', '2024-02-28', '2024-02-29', '2024-03-01',
                 '2024-04-09', '2024-04-12', '2024-04-13', '2024-04-15', '2024-04-16', '2024-12-30', '2024-12-31', '2025-01-01', '2025-01-08',
                 '2025-12-31']
SAMPLE_INFO = {}
for u in UNITS:
    keys = UNIT_KEYS[u]
    ref = REF[u]
    pan = pd.read_csv(FEAT_FILES[(u, 'panel')], dtype=str, keep_default_na=False, usecols=lambda c: c in set(keys) | {'training_eligible'})
    if len(pan) != len(ref) or not (pan[keys].to_numpy() == ref[keys].to_numpy()).all():
        raise DependencyError(f'{u}: panel keys differ from feature keys')
    rng = np.random.default_rng(int(EV['seed']))
    provs = [REG['provinces'][0], 'กรุงเทพมหานคร', REG['aliases'][next(iter(REG['aliases']))]]
    m_special = ref['target_date'].isin(SPECIAL_DATES).to_numpy() & ref['province_key'].isin(provs).to_numpy()
    wx_col = 'wx_rain_sum_mm_d2'
    wx_rows = np.flatnonzero(ref[wx_col].notna().to_numpy())
    pick = set(np.flatnonzero(m_special)) | set(rng.choice(len(ref), size=int(EV['builder_parity_random_rows_per_unit']), replace=False))
    pick |= set(rng.choice(wx_rows, size=min(200, len(wx_rows)), replace=False)) if len(wx_rows) else set()
    pick = np.array(sorted(pick))
    s_ref = ref.iloc[pick].reset_index(drop=True)
    elig = pan['training_eligible'].iloc[pick].to_numpy()
    built = build_feature_frame(REG, s_ref[keys].to_dict('records'), u)
    fp = compare_features(built, s_ref, [c for c in ALLOW[u] if c not in keys], CATS, TOL_B).assign(unit=u, sample='deterministic_grid_sample')
    FEAT_PARITY.append(fp)
    mu_b, _ = predict_frame(REG, built, u, baseline=False)
    mu_r, _ = predict_frame(REG, s_ref, u, baseline=False)
    SAMPLE_INFO[u] = {'n_rows': int(len(pick)), 'n_special_rows': int(m_special.sum()), 'n_eligible': int((elig == 'True').sum()),
                      'n_unknown_label': int((elig == 'False').sum()), 'years': sorted({d[:4] for d in s_ref['target_date']}),
                      'blocks': sorted(s_ref['time_block'].unique()) if u == '6h' else None, 'leap_day_included': bool(s_ref['target_date'].eq('2024-02-29').any()),
                      'rows_with_weather': int(s_ref[wx_col].notna().sum()), 'rows_without_weather': int(s_ref[wx_col].isna().sum()),
                      'rows_without_traffic': int(s_ref['trf_mean_aadt'].isna().sum()), 'rows_bank_calendar_unknown': int(s_ref['bank_calendar_known_asof'].eq(0).sum()),
                      'festival_rows': int((s_ref['fest_songkran_in'] + s_ref['fest_new_year_in']).gt(0).sum()),
                      'prediction_max_abs_diff_builder_vs_10': float(np.abs(mu_b - mu_r).max())}
    assert SAMPLE_INFO[u]['n_eligible'] > 0 and SAMPLE_INFO[u]['n_unknown_label'] > 0, f'{u}: sample lacks eligible or unknown-label cells'
    assert SAMPLE_INFO[u]['leap_day_included'] and SAMPLE_INFO[u]['rows_with_weather'] > 0 and SAMPLE_INFO[u]['rows_without_weather'] > 0
    assert u == 'daily' or SAMPLE_INFO[u]['blocks'] == REG['cfg']['request']['time_blocks']
    assert SAMPLE_INFO[u]['prediction_max_abs_diff_builder_vs_10'] <= TOL_R
FEAT_PARITY = pd.concat(FEAT_PARITY, ignore_index=True)
out_csv('builder_feature_parity.csv', FEAT_PARITY)
out_csv('builder_parity_sample.csv', pd.DataFrame([dict(unit=u, **{k: json.dumps(v, ensure_ascii=False) if isinstance(v, list) else v for k, v in s.items()})
                                                    for u, s in SAMPLE_INFO.items()]))
assert FEAT_PARITY['n_mismatch'].eq(0).all() and FEAT_PARITY['nan_pattern_equal'].all(), FEAT_PARITY[FEAT_PARITY['n_mismatch'] > 0]
passed('builder_feature_parity', f"{len(FEAT_PARITY)} feature x sample checks, 0 mismatches; samples {SAMPLE_INFO}")
print(PARITY.to_string(index=False))

PASS reload_parity_all_diagnostic_rows daily: 35467 rows, max |diff| selected 8.88e-16; 6h: 141868 rows, max |diff| selected 4.44e-16 (tol 1e-12)
PASS training_numeric_representation request features use the 10 -> 11 default CSV text/parse representation (TRAINING_NUMERIC_CONTRACT); doy_sin/doy_cos bitwise equal to the saved 10 features on every diagnostic row; {'daily': {'n_feature_values_changed_by_contract': 22590, 'n_feature_values_not_bitwise_equal_to_10': 0, 'max_abs_diff_raw_builder_vs_saved_11': 4.445917687922481}, '6h': {'n_feature_values_changed_by_contract': 90360, 'n_feature_values_not_bitwise_equal_to_10': 0, 'max_abs_diff_raw_builder_vs_saved_11': 1.434485481423968}}


PASS builder_feature_parity 124 feature x sample checks, 0 mismatches; samples {'daily': {'n_rows': 2253, 'n_special_rows': 57, 'n_eligible': 1889, 'n_unknown_label': 364, 'years': ['2021', '2022', '2023', '2024', '2025'], 'blocks': None, 'leap_day_included': True, 'rows_with_weather': 1244, 'rows_without_weather': 1009, 'rows_without_traffic': 571, 'rows_bank_calendar_unknown': 14, 'festival_rows': 59, 'prediction_max_abs_diff_builder_vs_10': 0.0}, '6h': {'n_rows': 2428, 'n_special_rows': 228, 'n_eligible': 1944, 'n_unknown_label': 484, 'years': ['2021', '2022', '2023', '2024', '2025'], 'blocks': ['00-06', '06-12', '12-18', '18-24'], 'leap_day_included': True, 'rows_with_weather': 1294, 'rows_without_weather': 1134, 'rows_without_traffic': 582, 'rows_bank_calendar_unknown': 23, 'festival_rows': 169, 'prediction_max_abs_diff_builder_vs_10': 0.0}}
 unit      feature_path                role                     method_id            split      n  max_abs_diff    tolerance  pass
daily save

## C. Stress เมื่อข้อมูลขาด

ตั้งกลุ่ม weather / traffic เป็น NaN บน keys เดิม แยก 2024 development / 2025 reused — เป็นการวัดความไว ไม่ใช่ผลเชิงสาเหตุ ไม่ใช้เลือกอะไร

In [6]:
# C. Missing-input stress on IDENTICAL saved evaluation keys (2024 development / 2025 reused kept separate). Perturbations set covariate groups to
#    NaN: they describe sensitivity to missing inputs, not physically coherent alternate worlds or causal effects; nothing is selected from them.
FG = IF10['feature_groups']
STRESS, STRESS_ROWS = [], {}
for u in UNITS:
    keys, e = UNIT_KEYS[u], REG['units'][u]
    dg, built = DIAG[u], DIAG[u + ':built']
    rows = dg[keys + ['split', 'y']].copy()
    preds = {}
    for scen, groups in EV['stress_scenarios'].items():
        mu, _ = predict_frame(REG, apply_missing(built, groups, FG), u, baseline=False)
        preds[scen] = mu
        rows[f'pred_{scen}'] = mu
    for scen, groups in EV['stress_scenarios'].items():
        used = sorted({c for g in groups for c in FG[g] if c in e['feature_columns']})
        for sp in sorted(dg['split'].unique()):
            m = dg['split'].eq(sp).to_numpy()
            mu, mu0 = preds[scen][m], preds['unmodified'][m]
            ms = FN11['metric_summary'](dg['y'].to_numpy(float)[m], mu, FLOOR)
            ms0 = FN11['metric_summary'](dg['y'].to_numpy(float)[m], mu0, FLOOR)
            STRESS.append({'unit': u, 'split': sp, 'scenario': scen, 'groups_set_missing': '+'.join(groups) or '(none)',
                           'model_columns_affected': len(used), 'n': int(m.sum()), 'all_finite_nonnegative': bool(np.isfinite(mu).all() and (mu >= 0).all()),
                           'mean_poisson_deviance': ms['mean_poisson_deviance'], 'mae': ms['mae'], 'sum_y': ms['sum_y'], 'sum_pred': ms['sum_pred'],
                           'delta_deviance_vs_unmodified': ms['mean_poisson_deviance'] - ms0['mean_poisson_deviance'], 'delta_mae_vs_unmodified': ms['mae'] - ms0['mae'],
                           'max_abs_pred_change': float(np.abs(mu - mu0).max()), 'mean_pred_ratio_vs_unmodified': float(mu.sum() / mu0.sum()),
                           'share_rows_already_missing_in_unmodified': float(built.loc[m, used].isna().any(axis=1).mean()) if used else None})
    STRESS_ROWS[u] = rows
    out_csv(f'stress_rows_{u}.csv.gz', rows, gz=True)
STRESS = pd.DataFrame(STRESS)
out_csv('stress_missing_inputs.csv', STRESS)
assert STRESS['all_finite_nonnegative'].all()
_unm = STRESS[STRESS['scenario'].eq('unmodified')]
assert (_unm['delta_deviance_vs_unmodified'] == 0).all()
passed('stress_missing_inputs', f'{len(STRESS)} unit x split x scenario rows on identical keys; every prediction finite and >= 0; '
       'descriptive only (no inference change selected from these scores)')
print(STRESS[['unit', 'split', 'scenario', 'model_columns_affected', 'n', 'mean_poisson_deviance', 'delta_deviance_vs_unmodified', 'max_abs_pred_change',
              'share_rows_already_missing_in_unmodified']].to_string(index=False))

# independent daily model vs sum of the four 6 h block predictions (diagnostic; never forced equal)
_rng = np.random.default_rng(int(EV['seed']))
_dates = sorted(_rng.choice(pd.date_range('2024-01-01', '2025-12-31').strftime('%Y-%m-%d').to_numpy(), size=24, replace=False).tolist())
RECON = daily_block_reconciliation(REG, REG['provinces'], _dates)
out_csv('daily_vs_block_sum_reconciliation.csv', RECON.describe(percentiles=[0.05, 0.5, 0.95]).T.reset_index().rename(columns={'index': 'statistic'}))
RECON_SUM = {'n_province_days': int(len(RECON)), 'sum_daily': float(RECON['daily_pred'].sum()), 'sum_blocks': float(RECON['sum_block_pred'].sum()),
             'median_ratio_blocks_to_daily': float(RECON['ratio_blocks_to_daily'].median()),
             'p05_ratio': float(RECON['ratio_blocks_to_daily'].quantile(0.05)), 'p95_ratio': float(RECON['ratio_blocks_to_daily'].quantile(0.95))}
passed('daily_vs_block_reconciliation_reported', f'{RECON_SUM} (independent models; difference reported, not forced to zero)')

PASS stress_missing_inputs 16 unit x split x scenario rows on identical keys; every prediction finite and >= 0; descriptive only (no inference change selected from these scores)
 unit            split                    scenario  model_columns_affected      n  mean_poisson_deviance  delta_deviance_vs_unmodified  max_abs_pred_change  share_rows_already_missing_in_unmodified
daily development_2024                  unmodified                       0   7885               1.094363                      0.000000             0.000000                                       NaN
daily reused_seen_2025                  unmodified                       0  27582               1.131936                      0.000000             0.000000                                       NaN
daily development_2024             weather_missing                       2   7885               1.094363                      0.000000             0.000000                                       1.0
daily reused_seen_2025        

## D. สัญญาคำขอ, negative controls ของ artifact และเวลาใช้งาน

In [7]:
# D. Request contract, tampering / missing-artifact negative controls (all must FAIL), order + alias + label-independence checks, and measured latency
import copy, tempfile, shutil
BKK = 'กรุงเทพมหานคร'
ALIAS_IN, ALIAS_OUT = next(iter(REG['aliases'].items()))
ok_d = {'province_key': BKK, 'target_date': '2024-05-01', 'unit': 'daily'}
ok_b = {'province_key': BKK, 'target_date': '2024-05-01', 'unit': '6h', 'time_block': '18-24'}
BAD = [('empty_batch', [], 'batch_size'), ('batch_129', [dict(ok_d, target_date=(dt.date(2024, 1, 1) + dt.timedelta(days=i)).isoformat()) for i in range(129)], 'batch_size'),
       ('not_a_list', 'x', 'invalid_type'), ('item_not_object', ['x'], 'invalid_type'),
       ('unknown_province', [dict(ok_d, province_key='Atlantis')], 'unknown_province'), ('province_whitespace', [dict(ok_d, province_key=' ' + BKK)], 'unknown_province'),
       ('province_not_string', [dict(ok_d, province_key=10)], 'invalid_type'),
       ('invalid_calendar_date', [dict(ok_d, target_date='2025-02-29')], 'invalid_date'), ('non_iso_format', [dict(ok_d, target_date='2024-1-05')], 'invalid_format'),
       ('datetime_string', [dict(ok_d, target_date='2024-05-01T00:00:00+07:00')], 'invalid_format'),
       ('thai_digits', [dict(ok_d, target_date='๒๐๒๔-05-01')], 'invalid_format'), ('buddhist_era', [dict(ok_d, target_date='2567-05-01')], 'buddhist_era_year'),
       ('bool_date', [dict(ok_d, target_date=True)], 'invalid_type'), ('nan_date', [dict(ok_d, target_date=float('nan'))], 'invalid_type'),
       ('inf_date', [dict(ok_d, target_date=float('inf'))], 'invalid_type'), ('int_date', [dict(ok_d, target_date=20240501)], 'invalid_type'),
       ('date_object', [dict(ok_d, target_date=dt.date(2024, 5, 1))], 'invalid_type'),
       ('before_service_range', [dict(ok_d, target_date='2023-12-31')], 'out_of_service_range'), ('after_service_range', [dict(ok_d, target_date='2031-01-01')], 'out_of_service_range'),
       ('unknown_unit', [dict(ok_d, unit='hourly')], 'invalid_unit'), ('6h_without_block', [{k: v for k, v in ok_b.items() if k != 'time_block'}], 'missing_field'),
       ('bad_block', [dict(ok_b, time_block='24-30')], 'invalid_time_block'), ('daily_with_block', [dict(ok_d, time_block='00-06')], 'incompatible_unit'),
       ('missing_date', [{'province_key': BKK, 'unit': 'daily'}], 'missing_field'), ('duplicate', [ok_d, dict(ok_d)], 'duplicate_request'),
       ('alias_duplicate_of_canonical', [dict(ok_d, province_key=ALIAS_IN), dict(ok_d, province_key=ALIAS_OUT)], 'duplicate_request')]
for f in ('y_recorded_count', 'training_eligible', 'split', 'wx_rain_sum_mm_d2', 'features', 'bundle_path', 'model_path', 'issue_time_local', 'method_id'):
    BAD.append((f'client_field_{f}', [dict(ok_d, **{f: 1})], 'unknown_field'))
CONTRACT = []
for name, req, code in BAD:
    try:
        predict_batch(REG, req)
        got = None
    except RequestValidationError as ex:
        got = sorted({e['code'] for e in ex.errors})
    CONTRACT.append({'case': name, 'expected_code': code, 'observed_codes': json.dumps(got), 'rejected': got is not None and code in got})
CONTRACT = pd.DataFrame(CONTRACT)
assert CONTRACT['rejected'].all(), CONTRACT[~CONTRACT['rejected']]
passed('request_validation_rejects', f'{len(CONTRACT)} invalid requests rejected with the expected error code (whole batch, before any feature build)')

# alias, order preservation, batch independence, label-independence, response fields
r_alias = predict_batch(REG, [dict(ok_d, province_key=ALIAS_IN)])['results'][0]
r_canon = predict_batch(REG, [dict(ok_d, province_key=ALIAS_OUT)])['results'][0]
assert r_alias['province_key'] == ALIAS_OUT and r_alias['alias_applied'] and r_alias['prediction'][QUANTITY] == r_canon['prediction'][QUANTITY]
_rng = np.random.default_rng(int(EV['seed']))
MIX = []
for i in range(64):
    u = UNITS[i % 2]
    q = {'province_key': REG['provinces'][int(_rng.integers(len(REG['provinces'])))], 'unit': u,
         'target_date': (dt.date(2024, 1, 1) + dt.timedelta(days=int(_rng.integers(0, 7 * 365)))).isoformat()}
    if u == '6h':
        q['time_block'] = CFG['request']['time_blocks'][i % 4]
    if q not in MIX:
        MIX.append(q)
res = predict_batch(REG, MIX)['results']
single = [predict_batch(REG, [q])['results'][0] for q in MIX]
perm = _rng.permutation(len(MIX))
res_p = predict_batch(REG, [MIX[i] for i in perm])['results']
assert [r['index'] for r in res] == list(range(len(MIX)))
assert all(r['request']['province_key'] == q['province_key'] and r['target_date'] == q['target_date'] and r['unit'] == q['unit'] for r, q in zip(res, MIX))
assert all(a['prediction'][QUANTITY] == b['prediction'][QUANTITY] for a, b in zip(res, single))
assert all(res_p[k]['prediction'][QUANTITY] == res[i]['prediction'][QUANTITY] for k, i in enumerate(perm))
for r in res:
    v = r['prediction'][QUANTITY]
    assert isinstance(v, float) and math.isfinite(v) and v >= 0 and r['research_status']['production_ready'] is False
    assert r['cell']['issue_time_local'].endswith('T18:00:00+07:00') and r['baseline_comparator']['label'].startswith('strongest')
    assert ('out_of_evaluated_period' in r['limitations']) == (r['target_date'] > CFG['request']['evaluated_period'][1])
    assert not any(k in json.dumps(r, ensure_ascii=False) for k in ('probability":', 'risk_level', 'colour', 'color'))
_pan = pd.read_csv(FEAT_FILES[('daily', 'panel')], dtype=str, keep_default_na=False, usecols=['province_key', 'target_date', 'training_eligible'])
_pan = _pan[_pan['target_date'].between('2024-01-01', '2025-12-31')]
_unk = _pan[_pan['training_eligible'].eq('False')].iloc[0]
_elg = _pan[_pan['training_eligible'].eq('True')].iloc[0]
r_lab = predict_batch(REG, [{'province_key': _unk['province_key'], 'target_date': _unk['target_date'], 'unit': 'daily'},
                            {'province_key': _elg['province_key'], 'target_date': _elg['target_date'], 'unit': 'daily'}])['results']
assert len(r_lab) == 2 and all(math.isfinite(r['prediction'][QUANTITY]) for r in r_lab)
EXAMPLE = predict_batch(REG, [ok_d, ok_b, {'province_key': ALIAS_IN, 'target_date': '2028-04-13', 'unit': 'daily'}])
passed('service_semantics', f'alias {ALIAS_IN}->{ALIAS_OUT}; {len(MIX)} mixed-unit requests keep input order and equal single-request and permuted-batch '
       'predictions exactly; unknown-label and eligible 2024/25 cells both served; out_of_evaluated_period flag matches the date')

# public predict_batch (validation -> request builder -> bundle + baseline) vs the SAVED notebook 11 diagnostics: daily only, 6h only, interleaved mixed
_TOL = float(EV['reload_parity_max_abs_diff'])
_PB = {}
for u in UNITS:
    _dg = DIAG[u]
    _idx = np.unique(np.r_[np.linspace(0, len(_dg) - 1, 60).round().astype(int), np.flatnonzero(_dg['target_date'].eq('2024-02-29').to_numpy())[:4]])
    _s = _dg.iloc[_idx].reset_index(drop=True)
    _PB[u] = (_s, [{k: r[k] for k in UNIT_KEYS[u]} | {'unit': u} for r in _s.to_dict('records')])
PB_PARITY = []


def _pb_parity(label, refs):
    """refs: [(unit, row of _PB[unit])] in batch order; every response row must echo its request and equal the saved 11 prediction."""
    batch = [_PB[u][1][j] for u, j in refs]
    out = predict_batch(REG, batch)['results']
    assert [r['index'] for r in out] == list(range(len(batch))), label
    assert [(r['province_key'], r['target_date'], r['unit'], r['time_block']) for r in out] == \
           [(q['province_key'], q['target_date'], q['unit'], q.get('time_block')) for q in batch], label
    for u in UNITS:
        k = [i for i, (uu, _) in enumerate(refs) if uu == u]
        if not k:
            continue
        s, e = _PB[u][0].iloc[[refs[i][1] for i in k]], REG['units'][u]
        for role, m, got in (('selected', e['method_id'], [out[i]['prediction'][QUANTITY] for i in k]),
                             ('baseline_comparator', e['baseline_method_id'], [out[i]['baseline_comparator'][QUANTITY] for i in k])):
            d = np.abs(np.asarray(got, dtype=float) - s[m].to_numpy(dtype=float))
            PB_PARITY.append({'batch': label, 'unit': u, 'role': role, 'method_id': m, 'n': len(k), 'max_abs_diff': float(d.max()), 'tolerance': _TOL,
                              'pass': bool((d <= _TOL).all())})


_pb_parity('daily_only', [('daily', j) for j in range(len(_PB['daily'][1]))])
_pb_parity('6h_only', [('6h', j) for j in range(len(_PB['6h'][1]))])
_pb_parity('mixed_interleaved_6h_first', [(u, j) for j in range(max(len(v[1]) for v in _PB.values())) for u in ('6h', 'daily') if j < len(_PB[u][1])])
PB_PARITY = pd.DataFrame(PB_PARITY)
out_csv('predict_batch_vs_saved11.csv', PB_PARITY)
assert PB_PARITY['pass'].all() and set(PB_PARITY['batch']) == {'daily_only', '6h_only', 'mixed_interleaved_6h_first'}, PB_PARITY[~PB_PARITY['pass']]
passed('public_predict_batch_matches_saved_11', f"{ {u: len(v[1]) for u, v in _PB.items()} } saved-11 diagnostic cells (incl. leap day) through predict_batch "
       f"daily-only, 6h-only and interleaved mixed: order kept, max |diff| {PB_PARITY['max_abs_diff'].max():.2e} (tol {_TOL})")

# tampering / missing artifacts / pinned code / versions: each must raise BEFORE any deserialization
LOADS = []
def _sentinel(p):
    LOADS.append(str(p))
    raise AssertionError('deserialization reached')
NEG = []
def _neg(name, fn):
    try:
        fn()
        NEG.append({'control': name, 'rejected': False, 'error': None})
    except DependencyError as ex:
        NEG.append({'control': name, 'rejected': True, 'error': str(ex)[:200]})
with tempfile.TemporaryDirectory() as td:
    td = Path(td)
    e = REG['units']['daily']
    tb = td / 'bundles' / 'daily' / e['bundle_path'].name
    tb.parent.mkdir(parents=True)
    shutil.copyfile(e['bundle_path'], tb)
    with open(tb, 'r+b') as fh:
        b0 = fh.read(1)
        fh.seek(0)
        fh.write(bytes([b0[0] ^ 0xFF]))
    _neg('tampered_bundle_byte', lambda: verified_joblib_load(tb, e['bundle_sha256'], 'tampered bundle', td, _sentinel))
    _neg('bundle_outside_trusted_root', lambda: verified_joblib_load(e['bundle_path'], e['bundle_sha256'], 'foreign root', td, _sentinel))
    _neg('missing_model_data_root', lambda: load_registry(BACKEND, CFG, dict(ROOT_OVERRIDES, model_data_dir=str(td / 'empty')), True, _sentinel))
    _cfg = copy.deepcopy(CFG)
    _cfg['trust_anchors']['model_manifest_sha256'] = '0' * 64
    _neg('model_manifest_anchor_mismatch', lambda: load_registry(BACKEND, _cfg, ROOT_OVERRIDES, True, _sentinel))
    _cfg2 = copy.deepcopy(CFG)
    _cfg2['expected_selection']['daily']['recommended_method'] = 'rf_poisson__full'
    _neg('selection_differs_from_expected', lambda: load_registry(BACKEND, _cfg2, ROOT_OVERRIDES, True, _sentinel))
    nb_t = json.loads(R['notebook11'].read_text('utf-8'))
    _fc = [c for c in nb_t['cells'] if c['cell_type'] == 'code' and _src(c).lstrip().startswith('# FUNCTIONS')][-1]
    _fc['source'] = _src(_fc) + '\nX_TAMPER = 1\n'
    (td / 'nb11_tampered.ipynb').write_text(json.dumps(nb_t), 'utf-8')
    _neg('tampered_notebook11_functions', lambda: exec_pinned_functions(td / 'nb11_tampered.ipynb', MAN['functions_cells_sha256'], 'nb11', ()))
    _neg('missing_notebook10', lambda: exec_pinned_functions(td / 'absent.ipynb', PM['functions_cells_sha256'], 'nb10', ()))
    _neg('incompatible_sklearn_version', lambda: check_versions(dict(REG['recorded_versions'], **{'scikit-learn': '0.0.0'}), CFG['version_policy']))
    _reg0 = load_registry(BACKEND, CFG, ROOT_OVERRIDES, load_models=False)
    _neg('registry_without_models', lambda: predict_batch(_reg0, [ok_d]))
NEG = pd.DataFrame(NEG)
assert NEG['rejected'].all() and not LOADS, (NEG[~NEG['rejected']], LOADS)
out_csv('contract_checks.csv', pd.concat([CONTRACT.assign(kind='request_validation'), NEG.rename(columns={'control': 'case'}).assign(kind='artifact_negative_control')],
                                         ignore_index=True))
passed('tamper_and_missing_artifacts_rejected', f'{len(NEG)} controls raised DependencyError; the deserializer was never reached ({len(LOADS)} calls)')

# measured latency (no thresholds invented): repeated full load (warm OS cache) and warm batches 1 / 32 / 128 per unit
_t = time.perf_counter()
load_registry(BACKEND, CFG, ROOT_OVERRIDES)
REPEAT_LOAD_S = time.perf_counter() - _t
BENCH = [{'kind': 'cold_registry_load', 'unit': '(both)', 'batch_size': None, **timing_summary([COLD_LOAD_S])},
         {'kind': 'repeat_registry_load_warm_os_cache', 'unit': '(both)', 'batch_size': None, **timing_summary([REPEAT_LOAD_S])}]
for u in UNITS:
    for size in EV['benchmark_batch_sizes']:
        req = []
        for i in range(size):
            q = {'province_key': REG['provinces'][i % len(REG['provinces'])], 'target_date': (dt.date(2024, 1, 1) + dt.timedelta(days=(i * 37) % 2557)).isoformat(), 'unit': u}
            if u == '6h':
                q['time_block'] = CFG['request']['time_blocks'][i % 4]
            req.append(q)
        predict_batch(REG, req)  # warm-up (excluded)
        ts = []
        for _ in range(int(EV['benchmark_repeats'])):
            _t = time.perf_counter()
            predict_batch(REG, req)
            ts.append(time.perf_counter() - _t)
        BENCH.append({'kind': 'warm_predict_batch', 'unit': u, 'batch_size': size, **timing_summary(ts)})
BENCH = pd.DataFrame(BENCH)
out_csv('latency_benchmark.csv', BENCH)
passed('latency_measured', f"cold load {COLD_LOAD_S:.2f}s; warm predict_batch medians (ms) "
       + ', '.join(f"{r.unit}/{int(r.batch_size)}={r.median_ms:.1f}" for r in BENCH.itertuples() if r.kind == 'warm_predict_batch') + ' (machine-specific; no threshold)')
print(BENCH.to_string(index=False))

PASS request_validation_rejects 35 invalid requests rejected with the expected error code (whole batch, before any feature build)


PASS service_semantics alias พระนครศรีอยุธยา->อยุธยา; 64 mixed-unit requests keep input order and equal single-request and permuted-batch predictions exactly; unknown-label and eligible 2024/25 cells both served; out_of_evaluated_period flag matches the date


PASS public_predict_batch_matches_saved_11 {'daily': 64, '6h': 64} saved-11 diagnostic cells (incl. leap day) through predict_batch daily-only, 6h-only and interleaved mixed: order kept, max |diff| 4.44e-16 (tol 1e-12)


PASS tamper_and_missing_artifacts_rejected 9 controls raised DependencyError; the deserializer was never reached (0 calls)


PASS latency_measured cold load 1.60s; warm predict_batch medians (ms) daily/1=18.8, daily/32=49.6, daily/128=111.1, 6h/1=18.6, 6h/32=49.7, 6h/128=110.6 (machine-specific; no threshold)
                              kind   unit  batch_size  n   median_ms      p95_ms      min_ms      max_ms
                cold_registry_load (both)         NaN  1 1599.766042 1599.766042 1599.766042 1599.766042
repeat_registry_load_warm_os_cache (both)         NaN  1  988.107209  988.107209  988.107209  988.107209
                warm_predict_batch  daily         1.0 15   18.783959   20.002567   18.002167   20.279125
                warm_predict_batch  daily        32.0 15   49.589375   53.869275   48.116667   54.797125
                warm_predict_batch  daily       128.0 15  111.102375  120.407421  108.171250  134.542959
                warm_predict_batch     6h         1.0 15   18.603333   19.390992   18.168042   19.890209
                warm_predict_batch     6h        32.0 15   49.685250   51.07613

## E. สรุปกลุ่มย่อย/จุดพลาดจาก notebook 12 ที่ตรวจรับแล้ว

In [8]:
# E. Critical subgroup / peak failure summary copied from the ACCEPTED notebook 12 analysis (hash-verified; nothing recomputed or re-selected)
AMAN_PATH = R['analysis_report_dir'] / 'analysis_manifest.json'
AMAN = read_json_strict(AMAN_PATH)
if AMAN.get('producer') != ANALYSIS_PRODUCER or AMAN.get('status') != 'completed' or AMAN.get('production_ready') is not False:
    raise DependencyError('notebook 12 analysis manifest is not a completed research-only run')
_mm = [i for i in AMAN['inputs'] if Path(i['path']).as_posix().endswith('reports/v2/models/model_manifest.json')]
if len(_mm) != 1 or _mm[0]['sha256'] != CFG['trust_anchors']['model_manifest_sha256']:
    raise DependencyError('notebook 12 analysed a different model manifest than the trust anchor')
A_OUT = {Path(o['path']).name: o['sha256'] for o in AMAN['outputs']}
A_FILES = {}
for u in UNITS:
    for n in (f'slices_{u}.csv', f'largest_residuals_{u}.csv'):
        A_FILES[n] = R['analysis_report_dir'] / n
        verify_file(A_FILES[n], A_OUT.get(n), f'12 output {n}')
passed('notebook12_outputs_verified', f'analysis manifest completed; {len(A_FILES)} slice/residual tables equal their recorded hashes; analysed model manifest = anchor')
K = int(EV['subgroup_top_k'])
SUB, PEAK = [], []
for u in UNITS:
    e = REG['units'][u]
    sl = pd.read_csv(A_FILES[f'slices_{u}.csv'], dtype={'level': str})
    sel = sl[sl['method_id'].eq(e['method_id'])]
    bl = sl[sl['method_id'].eq(e['baseline_method_id'])][['dimension', 'level', 'period', 'mean_poisson_deviance', 'mae']]
    j = sel.merge(bl, on=['dimension', 'level', 'period'], how='left', suffixes=('', '_baseline'))
    j['delta_deviance_vs_baseline'] = j['mean_poisson_deviance'] - j['mean_poisson_deviance_baseline']
    for per, g in j.groupby('period', sort=True):
        g = g[g['support'].eq('adequate_support') & g['mean_poisson_deviance'].notna()]
        worst = g.sort_values('mean_poisson_deviance', ascending=False).head(K).assign(reason='highest_selected_deviance')
        harm = g[g['delta_deviance_vs_baseline'] > 0].sort_values('delta_deviance_vs_baseline', ascending=False).head(K).assign(reason='selected_worse_than_baseline')
        SUB.append(pd.concat([worst, harm]).assign(unit=u))
    lr = pd.read_csv(A_FILES[f'largest_residuals_{u}.csv'], dtype=str, keep_default_na=False)
    lr = lr[lr['method_id'].eq(e['method_id'])].copy()
    lr['abs_residual'] = pd.to_numeric(lr['abs_residual'])
    for per, g in lr.groupby('period', sort=True):
        PEAK.append(g.sort_values('abs_residual', ascending=False).head(K))
SUB = pd.concat(SUB, ignore_index=True)[['unit', 'period', 'reason', 'dimension', 'level', 'n', 'sum_y', 'sum_pred', 'mean_poisson_deviance',
                                         'mean_poisson_deviance_baseline', 'delta_deviance_vs_baseline', 'mae', 'bias', 'support']]
PEAK = pd.concat(PEAK, ignore_index=True)
out_csv('subgroup_failure_summary.csv', SUB)
out_csv('peak_residual_summary.csv', PEAK)
passed('subgroup_peak_summary', f'{len(SUB)} adequate-support slice rows and {len(PEAK)} largest-residual rows from accepted 12 (descriptive; small groups excluded, not re-analysed)')
print(SUB.groupby(['unit', 'period', 'reason']).size().to_string())

PASS notebook12_outputs_verified analysis manifest completed; 4 slice/residual tables equal their recorded hashes; analysed model manifest = anchor
PASS subgroup_peak_summary 91 adequate-support slice rows and 48 largest-residual rows from accepted 12 (descriptive; small groups excluded, not re-analysed)
unit   period            reason                      
6h     development_2024  highest_selected_deviance       8
                         selected_worse_than_baseline    8
       oof               highest_selected_deviance       8
                         selected_worse_than_baseline    8
       reused_seen_2025  highest_selected_deviance       8
                         selected_worse_than_baseline    8
daily  development_2024  highest_selected_deviance       8
                         selected_worse_than_baseline    3
       oof               highest_selected_deviance       8
                         selected_worse_than_baseline    8
       reused_seen_2025  highest_selected_deviance

## F. Registry / ตัวอย่าง / รายงานไทย

In [9]:
# F. Registry / examples JSON, Thai report (numbers from the tables above only)
REGSUM = registry_summary(REG)
WRITTEN.append(write_json(REP / 'model_registry.json', REGSUM))
_bad_req = [dict(ok_d, y_recorded_count=3), dict(ok_b, time_block='24-30')]
try:
    predict_batch(REG, _bad_req)
    raise AssertionError('invalid example request was accepted')
except RequestValidationError as ex:
    _bad_errors = ex.errors
WRITTEN.append(write_json(REP / 'serving_examples.json', {'note': 'actual responses of predict_batch() in this run (research only)', 'requests': [ok_d, ok_b,
                          {'province_key': ALIAS_IN, 'target_date': '2028-04-13', 'unit': 'daily'}], 'response': EXAMPLE,
                          'validation_error_example': {'request': _bad_req, 'errors': _bad_errors}}))


def _f(x, nd=4):
    return 'n/a' if x is None or (isinstance(x, float) and not math.isfinite(x)) else (f'{x:.{nd}f}' if isinstance(x, float) else str(x))


L = ['# รายงาน 13 — สัญญาการให้บริการ inference V2 (research), registry ที่ตรวจ hash ได้, ความทนทานเมื่อข้อมูลขาด และเวลาใช้งาน', '',
     '**สถานะ: งานวิจัยเท่านั้น | production_ready = false | operational = BLOCKED** — ไม่ใช่ระบบเตือนภัย/ความเสี่ยงรายบุคคลหรือรายเส้นทาง ไม่มีข้อมูล live', '',
     '## 1. โมเดลที่ให้บริการ (ไม่ fit ใหม่ ไม่ tune ไม่เลือกใหม่)', '']
for u, s in REGSUM['units'].items():
    L.append(f"- **{u}**: `{s['method_id']}` (variant {s['variant']}, {s['n_features']} features, ใช้ weather={s['uses_weather']}, traffic={s['uses_traffic']}); "
             f"bundle sha256 `{s['bundle_sha256'][:16]}…`; training cutoff {s['training_cutoff_local']}; baseline เทียบ `{s['baseline_comparator']['method_id']}`")
L += ['', f"registry `{REG['registry_id']}` ตรวจ {len(REG['hashes'])} hash (trust anchors ใน `config/inference_v2.json`, รายงาน/อินพุต/โค้ด 11, selection lock, FUNCTIONS 10/11, "
      'ตาราง source 09, bundle) ก่อน `joblib.load` ทุกครั้ง; เวอร์ชัน scikit-learn/joblib ต้องตรงกับที่บันทึก', '',
      '## 2. สัญญาคำขอ (public function `predict_batch(reg, requests)`)', '',
      f"- รับเฉพาะ `province_key`, `target_date` (สตริง CE `YYYY-MM-DD`), `unit` (`daily`/`6h`) และ `time_block` (เฉพาะ 6h); batch 1–{CFG['request']['max_batch']}",
      f"- ช่วงวันที่ให้บริการ {CFG['request']['service_date_range'][0]}..{CFG['request']['service_date_range'][1]} เป็นการจำกัดทรัพยากรสำหรับ research scenario ไม่ใช่ช่วงที่ตรวจสอบแล้ว; "
      f"ช่วงที่ประเมิน (เคยเห็นแล้ว) {CFG['request']['evaluated_period'][0]}..{CFG['request']['evaluated_period'][1]}",
      '- issue time = วันก่อนเป้าหมาย 18:00 Asia/Bangkok (ลูกค้ากำหนดเองไม่ได้); ชื่อจังหวัดต้องตรง 77 key หรือ alias ที่ audit แล้วเท่านั้น '
      f"({', '.join(f'{k}→{v}' for k, v in REG['aliases'].items())})",
      f"- คำขอผิด {len(CONTRACT)} แบบถูกปฏิเสธทั้ง batch พร้อม error code (field ผลลัพธ์/mask/feature/path, วันที่ผิด/BE/bool/NaN/inf, block ไม่เข้ากับ unit, ซ้ำ ฯลฯ)",
      '- ผลลัพธ์: `expected_recorded_count` (ค่าเฉลี่ยจำนวน record ที่ถูกเลือก เป็นทศนิยม ไม่ปัดเศษ ไม่ใช่ความน่าจะเป็นของการชน ไม่มีสี/เปอร์เซ็นต์ความเสี่ยง), baseline comparator ที่ติดป้ายชัด, '
      'ขอบเขต cell/issue/timezone, hash โมเดล, coverage ของ source และ `limitations` (`out_of_evaluated_period`, `stale_model`, covariate ขาด ฯลฯ)', '',
      '## 3. ความตรงกัน (parity)', '']
for r in PARITY[PARITY['split'].eq('all')].to_dict('records'):
    L.append(f"- {r['unit']} {r['feature_path']} {r['role']} `{r['method_id']}`: n={r['n']}, max |diff| = {r['max_abs_diff']:.2e} "
             f"(tol {r['tolerance']:g}) → {'ผ่าน' if r['pass'] else 'ไม่ผ่าน'}")
for u, s in SAMPLE_INFO.items():
    L.append(f"- builder vs ตาราง feature 10 ({u}): {s['n_rows']} แถวกำหนดแน่นอน (วันพิเศษ {s['n_special_rows']}, eligible {s['n_eligible']}, unknown-label {s['n_unknown_label']}, "
             f"มี weather {s['rows_with_weather']}, ไม่มี weather {s['rows_without_weather']}, ไม่มี traffic {s['rows_without_traffic']}, ปี {', '.join(s['years'])}) — ไม่พบความต่าง")
L += ['', '**สัญญาการแทนค่าตัวเลขของ training (`training_numeric_representation`)**: feature ตัวเลขของคำขอผ่านการเขียนเป็นข้อความ CSV แบบ default ของ pandas '
      '(เหมือน `write_csv_gz` ของ notebook 10) และ parse ด้วย `pd.read_csv` แบบ default (เหมือน `load_unit_frame` ของ notebook 11) ก่อนเข้าโมเดล '
      'เพราะ parser default อาจให้ค่าต่างจากค่าในหน่วยความจำ 1 ulp (พบที่ doy_sin/doy_cos) และ threshold ของต้นไม้ขยายความต่างนี้ได้; '
      'ไม่ปัดทศนิยม ไม่ clip prediction ไม่เปลี่ยน tolerance ไม่ใช้ label หรือค่าที่บันทึกไว้']
for u, s in NUMREP_SUM.items():
    L.append(f"- {u}: ค่า feature ที่สัญญานี้เปลี่ยน {s['n_feature_values_changed_by_contract']} ค่า; ค่าที่ยังไม่เท่ากับตาราง 10 แบบ bitwise "
             f"{s['n_feature_values_not_bitwise_equal_to_10']} ค่า; ถ้าไม่ใช้สัญญานี้ prediction ต่างจาก 11 สูงสุด {s['max_abs_diff_raw_builder_vs_saved_11']:.3g} (diagnostic เท่านั้น)")
L += ['', '## 4. Stress: ตั้ง covariate เป็น missing บน keys เดิม (แยก 2024 development / 2025 reused)', '',
      '> เป็นการทดสอบความไวต่อข้อมูลขาดเท่านั้น ไม่ใช่โลกทางเลือกที่สอดคล้องทางกายภาพหรือผลเชิงสาเหตุ และไม่มีการเปลี่ยน inference จากคะแนนเหล่านี้', '',
      '| unit | split | scenario | คอลัมน์โมเดลที่ถูกกระทบ | n | deviance | Δ deviance | การเปลี่ยน prediction สูงสุด (ค่าสัมบูรณ์) | สัดส่วนแถวที่ขาดอยู่แล้ว |', '|---|---|---|---|---|---|---|---|---|']
for r in STRESS.itertuples():
    L.append(f'| {r.unit} | {r.split} | {r.scenario} | {r.model_columns_affected} | {r.n} | {_f(r.mean_poisson_deviance)} | {_f(r.delta_deviance_vs_unmodified)} | '
             f'{_f(r.max_abs_pred_change)} | {_f(r.share_rows_already_missing_in_unmodified, 3)} |')
L += ['', f"- daily กับผลรวม 4 ช่วง 6h เป็นโมเดลอิสระ: {RECON_SUM['n_province_days']} province-day, ratio median {RECON_SUM['median_ratio_blocks_to_daily']:.3f} "
      f"(p05 {RECON_SUM['p05_ratio']:.3f}, p95 {RECON_SUM['p95_ratio']:.3f}) — รายงานไว้ ไม่บังคับให้เท่ากัน", '',
      '## 5. ความปลอดภัยของ artifact', '', f'- {len(NEG)} negative controls (bundle ถูกแก้ไบต์, path นอก root, model data หาย, anchor ไม่ตรง, selection ต่าง, FUNCTIONS 11 ถูกแก้, '
      'notebook 10 หาย, เวอร์ชัน sklearn ไม่ตรง, registry ไม่มีโมเดล) ล้มเหลวตามคาดด้วย DependencyError ก่อนถึงขั้น deserialize', '',
      '## 6. เวลาใช้งานที่วัดจริง (เครื่องนี้; ไม่มีเกณฑ์ที่แต่งขึ้น)', '', '| ประเภท | unit | batch | n | median ms | p95 ms |', '|---|---|---|---|---|---|']
for r in BENCH.itertuples():
    L.append(f'| {r.kind} | {r.unit} | {"" if pd.isna(r.batch_size) else int(r.batch_size)} | {r.n} | {r.median_ms:.1f} | {r.p95_ms:.1f} |')
L += ['', '## 7. กลุ่มย่อย/จุดพลาดสูงสุดจาก notebook 12 ที่ตรวจรับแล้ว (support เพียงพอเท่านั้น)', '']
for (u, per, why), g in SUB.groupby(['unit', 'period', 'reason'], sort=True):
    top = '; '.join(f"{r.dimension}={r.level} (dev {_f(r.mean_poisson_deviance, 3)}, Δbaseline {_f(r.delta_deviance_vs_baseline, 3)}, n={r.n})" for r in g.head(3).itertuples())
    L.append(f'- {u} / {per} / {why}: {top}')
L += ['', '## 8. ข้อจำกัด (คงเดิมจากหลักฐาน V2)', ''] + [f'- {c}' for c in REGSUM['caveats']] + [
      f"- weather archive มีถึง {REG['coverage']['weather_last_feature_date']['wx_']} เท่านั้น: วันที่ให้บริการทั้งหมด (≥2024) ไม่มีค่า weather โมเดล daily จึงทำงานด้วย weather = NaN เสมอ",
      f"- traffic เป็น AADT รายปี (survey ปี {REG['coverage']['traffic_survey_years_ce']}); ปีอนาคตใช้ vintage ล่าสุดก่อนหน้าที่มี ณ issue time และติดป้าย `traffic_vintage_stale` เมื่อห่างเกิน 1 ปี",
      f"- bank calendar ที่ review แล้วมีเฉพาะปี {REG['coverage']['bank_calendar_reviewed_years']}; ปีอื่นเป็น unknown (NaN) ไม่ถือเป็นวันทำการปกติ",
      '- ไม่มี untouched test; ผลนี้ปิดงาน ML research serving เท่านั้น ไม่ได้แก้ blocker ทางวิทยาศาสตร์สำหรับ production']
(REP / 'serving_report_th.md').write_text('\n'.join(L) + '\n', 'utf-8')
WRITTEN.append(REP / 'serving_report_th.md')
print('\n'.join(L[:40]))

# รายงาน 13 — สัญญาการให้บริการ inference V2 (research), registry ที่ตรวจ hash ได้, ความทนทานเมื่อข้อมูลขาด และเวลาใช้งาน

**สถานะ: งานวิจัยเท่านั้น | production_ready = false | operational = BLOCKED** — ไม่ใช่ระบบเตือนภัย/ความเสี่ยงรายบุคคลหรือรายเส้นทาง ไม่มีข้อมูล live

## 1. โมเดลที่ให้บริการ (ไม่ fit ใหม่ ไม่ tune ไม่เลือกใหม่)

- **daily**: `hgb_poisson__full` (variant full, 30 features, ใช้ weather=True, traffic=True); bundle sha256 `1b7e794d6278cd6c…`; training cutoff 2023-12-31T18:00:00+07:00; baseline เทียบ `baseline_province_month_k30`
- **6h**: `hgb_poisson__calendar_traffic` (variant calendar_traffic, 29 features, ใช้ weather=False, traffic=True); bundle sha256 `f006ded45f660be8…`; training cutoff 2023-12-31T18:00:00+07:00; baseline เทียบ `baseline_province_block_mean`

registry `2f8a1e2e910fcd01` ตรวจ 32 hash (trust anchors ใน `config/inference_v2.json`, รายงาน/อินพุต/โค้ด 11, selection lock, FUNCTIONS 10/11, ตาราง source 09, bundle) ก่อน `joblib.load` ทุกครั้ง; เวอร์ชัน 

## G. ตรวจขั้นสุดท้ายและ manifest

In [10]:
# G. Final checks: read-only inputs unchanged, strict JSON, then the serving manifest (written last; an existing one is never overwritten)
SNAP_AFTER = {p: sha256_file(p) for p in SNAP_BEFORE}
_changed = sorted(k for k in SNAP_BEFORE if SNAP_BEFORE[k] != SNAP_AFTER.get(k))
assert not _changed, f'read-only inputs changed during 13: {_changed}'
passed('read_only_inputs_unchanged', f'{len(SNAP_AFTER)} files identical before/after')
for p in WRITTEN:
    if str(p).endswith('.json'):
        read_json_strict(p)
passed('strict_json_outputs', 'every JSON output re-read with NaN/Infinity rejected')
FN13_SHA = sha256_bytes(functions_cells_source(NB13).encode('utf-8'))
if (REP / 'serving_manifest.json').exists():
    raise RuntimeError('serving_manifest.json appeared during this run; refusing to overwrite')
SMAN = {'schema': SERVING_SCHEMA, 'producer': SERVING_PRODUCER, 'status': 'completed', 'status_note': 'research serving contract; passing checks is not acceptance',
        'production_ready': False, 'operational_status': 'BLOCKED', 'research_serving_ready': True, 'started_utc': T_RUN0, 'finished_utc': utc_now(),
        'functions_cells_sha256': FN13_SHA, 'code_cells_sha256': code_cells_sha256(NB13), 'config_sha256': sha256_file(CFG_PATH),
        'registry_id': REG['registry_id'], 'verified_input_hashes': REG['hashes'], 'runner_logs_11': REG['runner_logs'],
        'evaluation_inputs': {f'{u}:{p}': {'path': str(v), 'sha256': SNAP_BEFORE[str(v)]} for (u, p), v in FEAT_FILES.items()}
                             | {f'{u}:diagnostic_predictions': {'path': str(v), 'sha256': SNAP_BEFORE[str(v)]} for u, v in DIAG_FILES.items()},
        'analysis_manifest_sha256': SNAP_BEFORE[str(AMAN_PATH)], 'versions': VERSIONS, 'recorded_bundle_versions': REG['recorded_versions'],
        'threads': REG['threads'], 'roots': {k: str(v) for k, v in R.items()}, 'output_root': str(OUT),
        'public_interface': {
            'load': 'load_registry(backend=None, cfg=None, roots=None, load_models=True, loader=None) -> registry dict (verifies every hash before joblib.load)',
            'predict': 'predict_batch(reg, requests, include_baseline=True) -> {schema, registry_id, research_only, production_ready, operational_status, n_results, seconds, results[]}',
            'validate': 'validate_requests(reg, requests) -> normalized cells | raises RequestValidationError(.errors=[{index, field, code, message}])',
            'metadata': 'registry_summary(reg) -> JSON-safe registry (methods, features, cutoff, coverage, contract, caveats)',
            'diagnostic': 'daily_block_reconciliation(reg, province_keys, dates) -> DataFrame (independent models, not forced equal)',
            'features': 'build_feature_frame(reg, cells, unit) = training_numeric_representation(build_raw_feature_frame(...)); only this path feeds a bundle',
            'training_numeric_contract': TRAINING_NUMERIC_CONTRACT,
            'errors': 'RequestValidationError = client error (whole batch rejected); DependencyError = server-side trust / artifact failure (no fallback)',
            'how_to_load': "exec ONLY the two '# FUNCTIONS' cells of notebooks/13_inference_robustness_contract.ipynb after checking their sha256 equals functions_cells_sha256 here"},
        'results': {'parity': PARITY.to_dict('records'), 'public_predict_batch_vs_11': PB_PARITY.to_dict('records'),
                    'training_numeric_representation': {'summary': NUMREP_SUM, 'per_feature': NUMREP.to_dict('records')}, 'builder_samples': SAMPLE_INFO, 'stress': STRESS.to_dict('records'), 'daily_vs_blocks': RECON_SUM,
                    'latency': BENCH.to_dict('records'), 'n_request_contract_cases': int(len(CONTRACT)), 'n_artifact_negative_controls': int(len(NEG))},
        'checks': CHECKS, 'outputs': [{'path': str(Path(p).relative_to(OUT)) if inside(p, OUT) else str(p), 'sha256': sha256_file(p)} for p in WRITTEN],
        'caveats': REGSUM['caveats']}
write_json(REP / 'serving_manifest.json', SMAN)
read_json_strict(REP / 'serving_manifest.json')
print(f"serving manifest -> {REP / 'serving_manifest.json'}; {len(CHECKS)} checks passed; production_ready False (research only)")

PASS read_only_inputs_unchanged 15 files identical before/after
PASS strict_json_outputs every JSON output re-read with NaN/Infinity rejected
serving manifest -> /Users/earthnaja/Documents/GitHub/Work - Earth/Project Work/Source Code/RoadAware/backend/reports/v2/serving/serving_manifest.json; 18 checks passed; production_ready False (research only)


## ข้อจำกัด

- ไม่มี untouched test; ปี 2024/2025 เป็น diagnostic ที่เคยเห็นแล้ว
- weather archive ไม่มีปี 2024/2025: ทุกวันที่ให้บริการไม่มีค่า weather
- ผลนี้ปิดงาน ML research serving เท่านั้น ไม่ใช่การอนุญาตใช้งานเชิงปฏิบัติการ